---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

---

## Group 24 - AyamNutellaBear

- Geraldo Artemius 13524005
- Mikhael Yonatan 13524051
- Junior Natra Situmorang 13524055
- Reynard Nathanael 13524103
- Nicholas Luis Chandra 13524105

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

Ronaldo Hotel ingin memperkirakan berapa banyak reservasi yang akan tetap berlangsung agar dapat merencanakan ketersediaan kamar. Setiap reservasi menjadi satu contoh klasifikasi biner. Label `is_canceled` bernilai 1 jika reservasi dibatalkan dan 0 jika reservasi tetap berlangsung.

## Aim

Tujuan kelompok adalah membangun dan mengevaluasi lima algoritma *supervised learning* dari awal untuk memprediksi pembatalan reservasi hotel. Kelima algoritma tersebut adalah K-Nearest Neighbors (KNN), Decision Tree Learning (DTL), Logistic Regression, Support Vector Machine (SVM), dan Naive Bayes. Kinerja setiap implementasi akan dibandingkan dengan padanannya di scikit-learn pada reservasi validasi yang sama. Model buatan kelompok kemudian digunakan untuk menghasilkan prediksi data uji bagi kompetisi Kaggle.

## Metric

Metrik utama adalah **Macro F1**, yaitu rata-rata F1 kelas tidak batal dan F1 kelas batal. Metrik ini sama dengan metrik kompetisi Kaggle. Karena hanya sekitar 32,8% reservasi yang batal, akurasi bisa tinggi walaupun banyak pembatalan terlewat; Macro F1 memberi bobot yang sama pada kedua kelas sehingga kesalahan pada kelas batal tetap terhitung. Untuk model yang menghasilkan peluang, ambang prediksi ikut disetel agar Macro F1 maksimal.

## Dataset

- `train.csv` berisi 27.373 reservasi berlabel, termasuk 8.969 reservasi yang batal.
- `test.csv` berisi 6.844 reservasi yang perlu diprediksi. Kedua berkas memiliki 17 fitur yang mencakup informasi tamu, rincian reservasi, tanggal kedatangan, dan riwayat pemesanan.
- `sample_submission.csv` menunjukkan format keluaran dengan kolom `id` dan `is_canceled`. Kolom `id` menghubungkan prediksi dengan reservasinya.

## Approach

1. Data berlabel dibagi stratified menjadi 80% `train_raw` dan 20% `val_raw` (holdout). Satu set 3-Fold stratified pada `train_raw` (`splits`) dipakai untuk semua tuning, sehingga semua model dibandingkan pada fold yang sama.
2. Setiap model punya satu fungsi `prepare_features_*` yang mempelajari statistik hanya dari data fit, lalu menerapkannya ke data evaluasi. Fungsi yang sama dipakai untuk K-Fold, holdout, dan submission.
3. Setiap model dilatih dulu dengan konfigurasi awal (Modeling and Validation), lalu disetel dengan K-Fold di bagian Improvements. Konfigurasi final setiap model dipilih dari skor K-Fold, dan `val_raw` hanya dipakai sebagai pemeriksaan akhir.
4. Model dengan Macro F1 K-Fold tertinggi dilatih ulang pada seluruh `train.csv` untuk membuat submission. Temuan utama: banyak pemesanan dibuat bersamaan ("kerabat") dan labelnya cenderung sama, yang dimanfaatkan oleh KNN kerabat.

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [ ]:
pip install -q numpy pandas matplotlib seaborn scipy scikit-learn

## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import KBinsDiscretizer
from scipy.stats import chi2_contingency           
from sklearn.metrics import mutual_info_score
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import KFold
import time
from copy import deepcopy
from collections import namedtuple

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

## Settings

Paths and shared constants are kept in this class.

In [ ]:
import shutil

ON_KAGGLE = Path('/kaggle/input').exists()
COMPETITION = 'T2-IF3170-2026-2027'

def find_src_dir():
    # Laptop: folder src di repo. Kaggle: salin folder yang berisi knn.py (apa pun namanya) menjadi 'algorithm'
    if not ON_KAGGLE:
        return Path('..').resolve()
    knn_files = list(Path('/kaggle/input').rglob('knn.py'))
    if not knn_files:
        raise FileNotFoundError('knn.py tidak ditemukan di /kaggle/input. Tambahkan dataset kode algoritma lewat Add Input.')
    if not (knn_files[0].parent / 'knn_kerabat.py').exists():
        raise FileNotFoundError('knn_kerabat.py belum ada di dataset kode algoritma. Upload sebagai New Version.')
    target = Path('/kaggle/working/src/algorithm')
    shutil.copytree(knn_files[0].parent, target, dirs_exist_ok=True)
    return target.parent

def find_data_dir(src_dir):
    # 1) src/data di repo, 2) input Kaggle, 3) unduh dengan kagglehub (butuh Internet menyala)
    local = src_dir / 'data'
    if (local / 'train.csv').exists():
        return local
    if ON_KAGGLE:
        for p in Path('/kaggle/input').rglob('train.csv'):
            if (p.parent / 'test.csv').exists():
                return p.parent
    import kagglehub
    return Path(kagglehub.competition_download(COMPETITION))

class Settings:
    SEED   = 42
    TARGET = 'is_canceled'
    ID_COL = 'id'
    VAL_SIZE = 0.2
    FEATURE_ENGINEERING = True
    SRC_DIR    = find_src_dir()
    DATA_DIR   = find_data_dir(SRC_DIR)
    OUTPUT_DIR = Path('/kaggle/working') if ON_KAGGLE else SRC_DIR.parent / 'output'

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(CFG.SRC_DIR) not in sys.path:
    sys.path.append(str(CFG.SRC_DIR))

TARGET, ID_COL = CFG.TARGET, CFG.ID_COL
print('Kaggle   :', ON_KAGGLE)
print('Data     :', CFG.DATA_DIR)
print('Algoritma:', CFG.SRC_DIR / 'algorithm')
print('Output   :', CFG.OUTPUT_DIR)

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_df   = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df    = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})
sample_sub = pd.read_csv(CFG.DATA_DIR / 'sample_submission.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}  Sample submission: {sample_sub.shape}')

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
summary = pd.DataFrame({
    'dtype': train_df.dtypes.astype(str),
    'unique': train_df.nunique(),
    'missing': train_df.isna().sum(),
    'min': train_df.apply(lambda s: s.dropna().min()),
    'max': train_df.apply(lambda s: s.dropna().max()),
    'value': [train_df[c].dropna().unique()[:5].tolist() for c in train_df.columns],
})
summary

### Klasifikasi Kolom

In [ ]:
NUMERICAL   = ['num_adults', 'num_children', 'weekend_nights', 'weekday_nights', 'days_before_arrival', 'prior_cancellations', 'prior_completed_bookings', 'nightly_rate_idr', 'special_requests']
CATEGORICAL = ['meal_package', 'room_category', 'booking_channel']
DATE        = ['arrival_year', 'arrival_month', 'arrival_day']
BINARY      = ['needs_parking', 'is_returning_guest']
FEATURES    = NUMERICAL + CATEGORICAL + DATE + BINARY

assert set(FEATURES) | {ID_COL, TARGET} == set(train_df.columns)
assert set(train_df.columns) - set(test_df.columns) == {TARGET}

**Insights**

- Dataset train berisi 27.373 baris dan test 6.844 baris.
- `id` unik di setiap baris dan hanya digunakan untuk file submission, bukan sebagai fitur.
- Kolom dikelompokkan menjadi numerical, categorical, date, dan binary.

## Explore Data

This part covers distributions and relationships between attributes.

In [ ]:
# Distribusi target
print(train_df[TARGET].value_counts())
print(train_df[TARGET].value_counts(normalize=True).round(3))

In [ ]:
train_df[NUMERICAL + DATE].describe().T

In [ ]:
# Distribusi nilai fitur kategorikal dan biner
for c in CATEGORICAL + BINARY:
    print(f'\n{c}:')
    print(train_df[c].value_counts(dropna=False))

**Insights**

- Distribusi target 67% tidak cancel dan 33% cancel.
- Fitur numerik cenderung right skewed atau positive skewed. Kolom `prior_cancellations` dan `prior_completed_bookings` kebanyakan bernilai 0.
- Ada kategori yang langka: `Full Board` (4 baris) dan `Type E` (7 baris).

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
missing = pd.DataFrame({
    'train_missing': train_df.isna().sum(),
    'train_%': (train_df.isna().mean() * 100).round(2),
    'test_missing': test_df.isna().sum(),
    'test_%': (test_df.isna().mean() * 100).round(2),
})
missing[(missing['train_missing'] > 0) | (missing['test_missing'] > 0)]

In [ ]:
print('Id yang duplikat di train  :', train_df[ID_COL].duplicated().sum())
print('Data yang duplikat di train:', train_df.drop(columns=[ID_COL]).duplicated().sum())

In [ ]:
check = {
    'Ga ada tamu sama sekali': (train_df['num_adults'] == 0) & (train_df['num_children'] == 0),
    'Ga nginep': (train_df['weekend_nights'] + train_df['weekday_nights']) == 0,
    'Harganya < 0': train_df['nightly_rate_idr'] < 0,
    'Harganya 0 tapi bukan Complimentary': (train_df['nightly_rate_idr'] == 0) & (train_df['booking_channel'] != 'Complimentary'),
    'days_before_arrival < 0': train_df['days_before_arrival'] < 0,
}

for name, mask in check.items():
    print(f'{name:40s}: {mask.sum()}')

dates = pd.to_datetime(dict(year=train_df['arrival_year'], month=train_df['arrival_month'], day=train_df['arrival_day']), errors='coerce')
print('Data tanggal tidak valid:', dates.isna().sum())

In [ ]:
# Deteksi outlier 
for c in NUMERICAL:
    q1, q3 = train_df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    batas_bawah = q1 - 1.5 * iqr
    batas_atas  = q3 + 1.5 * iqr
    n_outlier = ((train_df[c] < batas_bawah) | (train_df[c] > batas_atas)).sum()
    print(f'{c:26s} IQR={iqr:>10.1f}  batas=[{batas_bawah:.1f}, {batas_atas:.1f}]  outlier={n_outlier}')

In [ ]:
print(train_df['num_children'].value_counts().sort_index())
print(train_df['weekday_nights'].value_counts().sort_index().tail())

**Insights**

- Missing values ada di 8 kolom (sekitar 3–11%) dengan proporsi yang mirip di train dan test.
- Duplikasi tidak ada `id` ganda, tetapi ada 64 baris yang identik di semua kolom lain.
- Harga 0 tetapi bukan Complimentary: 146 baris.
- Tidak ada tamu 0, harga negatif, `days_before_arrival` negatif, ataupun tanggal yang tidak valid.
- Kolom dengan IQR = 0 (`num_adults`, `num_children`, `prior_cancellations`, `prior_completed_bookings`) tidak dapat dinilai dengan IQR, karena semua nilai selain nilai paling umum akan dianggap outlier. 
- Ada nilai yang tidak realistis yaitu `num_children` = 9 dan 10, yang melompat jauh dari nilai 3.

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
train_set, val_set = train_test_split(
    train_df,
    test_size=CFG.VAL_SIZE,
    stratify=train_df[TARGET],
    random_state=CFG.SEED,
)

train_set = train_set.reset_index(drop=True)
val_set   = val_set.reset_index(drop=True)
test_set  = test_df.copy()
train_raw = train_set.copy()
val_raw = val_set.copy()

print(f'Train: {train_set.shape}  Val: {val_set.shape}  Test: {test_set.shape}')
print('Proporsi target train:', train_set[TARGET].value_counts(normalize=True).round(3).to_dict())
print('Proporsi target val  :', val_set[TARGET].value_counts(normalize=True).round(3).to_dict())

In [ ]:
# 3-Fold stratified yang dipakai SEMUA tuning dan seleksi pada notebook ini
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=CFG.SEED)
splits = list(folds.split(train_raw, train_raw[TARGET]))
y_val = val_raw[TARGET].to_numpy(dtype=int)
print('Ukuran fold evaluasi:', [len(eval_idx) for _, eval_idx in splits])

**Insights**

- Split dilakukan secara *stratified* (stratified random sampling) sehingga proporsi kelas 67:33 sama di train dan validation.
- Semua statistik pada tahap Data Preparation dihitung dari `train_set` saja.

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [ ]:
print('Jumlah fitur yang dipakai:', len(FEATURES))
print('Fitur:', FEATURES)
print('Bukan fitur:', [ID_COL, TARGET])

**Insights**

- Semua 17 fitur digunakan. 
- `id` tidak dipakai sebagai fitur karena hanya pengenal unik, tetapi tetap disimpan untuk file submission.
- Tidak dilakukan sampling karena ukuran data masih kecil untuk diproses. Seleksi record hanya berupa penghapusan duplikat di `train_set`.

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Duplicates

This part covers repeated records.

In [ ]:
before = len(train_set)

train_set = train_set.drop_duplicates(subset=train_set.columns.drop(ID_COL), keep='first').reset_index(drop=True)

print(f'Duplikat dihapus dari train_set: {before - len(train_set)}')
print(f'Jumlah baris train_set sekarang: {train_set.shape}')

**Insights**

- Duplikat dihapus karena baris yang identik di semua kolom (selain `id`) membuat pola tertentu tercatat lebih dari sekali. Baris yang duplikat di drop agar tidak menimbulkan bias pada hasil analisis dan tidak membuat overfitting.
- Penghapusan hanya dilakukan di `train_set`. 

### Missing Values

This part covers missing values, if any.

In [ ]:
NUMERICAL_MISSING   = ['num_adults', 'num_children', 'days_before_arrival', 'nightly_rate_idr', 'special_requests']
CATEGORICAL_MISSING = ['meal_package', 'room_category', 'needs_parking']

fill = {c: train_set[c].median() for c in NUMERICAL_MISSING}
fill.update({c: train_set[c].mode().iloc[0] for c in CATEGORICAL_MISSING})
print(fill)

train_set = train_set.fillna(fill)
val_set   = val_set.fillna(fill)
test_set  = test_set.fillna(fill)

for name, df in [('train', train_set), ('val', val_set), ('test', test_set)]:
    print(f'Sisa missing {name}: {df.isna().sum().sum()}')

**Insights**

- Data Numerik diisi median. Jika pakai mean, akan ikut tertarik oleh nilai ekstrem, sedangkan median tidak.
- Data Kategorikal dan biner diisi modus. Karena string tidak memiliki rata-rata atau nilai tengah. 

### Pemesanan Kerabat

Dua pemesanan disebut **kerabat** jika tanggal kedatangan, jumlah malam weekend, jumlah malam weekday, dan kanal pemesanannya sama. Analisis memakai `train_raw` saja. Untuk melihat seberapa sering data baru punya kerabat di data latih, `val_raw` dipakai sebagai pengganti data test (data test tidak disentuh).

In [ ]:
KUNCI_KERABAT = ['arrival_year', 'arrival_month', 'arrival_day', 'weekend_nights', 'weekday_nights', 'booking_channel']
KUNCI_LONGGAR = ['arrival_year', 'arrival_month', 'arrival_day', 'booking_channel']

grup = train_raw.groupby(KUNCI_KERABAT)[TARGET].agg(['size', 'mean'])
grup_besar = grup[grup['size'] >= 2]
label_seragam = ((grup_besar['mean'] == 0) | (grup_besar['mean'] == 1))

# Untuk setiap baris val: apakah punya kerabat di train_raw, dan apakah mayoritas label kerabatnya sama
val_kerabat = val_raw[KUNCI_KERABAT + [TARGET]].merge(
    grup.rename(columns={'size': 'n_kerabat', 'mean': 'rasio_kerabat'}).reset_index(), on=KUNCI_KERABAT, how='left')
punya = val_kerabat['n_kerabat'].notna()
tebakan = (val_kerabat.loc[punya, 'rasio_kerabat'] >= 0.5).astype(int)

analisis_kerabat = pd.Series({
    'Baris train_raw yang punya kerabat (%)': 100 * grup_besar['size'].sum() / len(train_raw),
    'Kelompok kerabat (ukuran >= 2)': len(grup_besar),
    'Kelompok dengan label seragam (%)': 100 * label_seragam.mean(),
    'Baris val_raw yang punya kerabat di train_raw (%)': 100 * punya.mean(),
    'Akurasi tebakan "ikuti mayoritas kerabat" pada baris tsb (%)': 100 * (tebakan == val_kerabat.loc[punya, TARGET]).mean(),
    'Tingkat batal: punya kerabat (%)': 100 * val_kerabat.loc[punya, TARGET].mean(),
    'Tingkat batal: tanpa kerabat (%)': 100 * val_kerabat.loc[~punya, TARGET].mean(),
})
analisis_kerabat.round(2)

**Insights**

- 91,0% baris `train_raw` punya kerabat (3.175 kelompok berukuran ≥ 2), dan 90,5% baris `val_raw` punya kerabat di `train_raw`. Pola ini juga hampir pasti ada pada data test.
- Hanya 49,4% kelompok yang labelnya seragam, dan aturan "ikuti mayoritas kerabat" benar 72,8% pada baris yang punya kerabat. Jadi kerabat adalah petunjuk yang berguna, tetapi tidak cukup dipakai sendirian; petunjuk ini perlu digabung dengan fitur lain.
- Tingkat batal pemesanan yang punya kerabat 33,9%, sedangkan yang tidak punya kerabat hanya 21,8%.

### Imputasi `days_before_arrival` dari Kerabat

Sebelumnya `days_before_arrival` yang kosong diisi median seluruh data. Karena kerabat biasanya dipesan pada hari yang sama, lead time kerabat adalah tebakan yang jauh lebih baik. `ImputasiLeadKerabat` mencari median lead time kerabat secara bertingkat:
1. kunci kerabat lengkap (tanggal kedatangan, malam weekend, malam weekday, kanal),
2. tanggal kedatangan dan kanal,
3. tanggal kedatangan saja.

Yang tidak punya kerabat sama sekali tetap kosong, lalu diisi median oleh `clean_reservations` seperti biasa. Tabel median dipelajari dari data fit saja. Sel kedua menguji kualitasnya: 1.000 lead time yang diketahui disembunyikan, lalu ditebak.

In [ ]:
class ImputasiLeadKerabat:
    TINGKAT = [KUNCI_KERABAT, KUNCI_LONGGAR, ['arrival_year', 'arrival_month', 'arrival_day']]

    def fit(self, df):
        diketahui = df[df['days_before_arrival'].notna()]
        self.tabel_ = [diketahui.groupby(kunci)['days_before_arrival'].median() for kunci in self.TINGKAT]
        return self

    def transform(self, df):
        df = df.copy()
        lead = df['days_before_arrival'].to_numpy(dtype=float).copy()
        for kunci, tabel in zip(self.TINGKAT, self.tabel_):
            kosong = np.isnan(lead)
            if not kosong.any():
                break
            posisi = pd.MultiIndex.from_frame(df.loc[kosong, kunci]) if len(kunci) > 1 else df.loc[kosong, kunci[0]]
            lead[kosong] = tabel.reindex(posisi).to_numpy()
        df['days_before_arrival'] = np.round(lead)
        return df

In [ ]:
diketahui = train_raw[train_raw['days_before_arrival'].notna()]
uji_imputasi = diketahui.sample(1000, random_state=CFG.SEED)
latih_imputasi = train_raw.drop(index=uji_imputasi.index)

imputer_uji = ImputasiLeadKerabat().fit(latih_imputasi)
tebakan_lead = imputer_uji.transform(uji_imputasi.assign(days_before_arrival=np.nan))['days_before_arrival']
median_lead = latih_imputasi['days_before_arrival'].median()
tertebak = tebakan_lead.notna()

asli = uji_imputasi['days_before_arrival']
pd.Series({
    'Baris yang punya kerabat (%)': 100 * tertebak.mean(),
    'MAE imputasi kerabat (hari)': (tebakan_lead.fillna(median_lead) - asli).abs().mean(),
    'MAE imputasi median (hari)': (median_lead - asli).abs().mean(),
}).round(2)

**Insights**

- Pada 1.000 lead time yang sengaja disembunyikan, error rata-rata imputasi kerabat 37,8 hari, sedangkan imputasi median 61,1 hari (sekitar 38% lebih kecil). Hampir semua baris (99,9%) punya kerabat di salah satu tingkat kunci.
- Lead time adalah fitur terpenting untuk pembatalan, sehingga imputasi yang lebih akurat juga memperbaiki `booking_ordinal` (tanggal pemesanan).
- Imputasi ini hanya dipakai oleh pipeline kerabat (KNN dan DTL final). Pipeline model lain tetap memakai median seperti sebelumnya.

### Outliers

This part covers values far from the rest of the data.

In [ ]:
def replace_child_outliers(df, median):
    df = df.copy()
    df.loc[df['num_children'] > 3, 'num_children'] = median
    return df

child_median = train_set['num_children'].median()   

for name, df in [('train', train_set), ('val', val_set), ('test', test_set)]:
    print(f'{name} num_children > 3: {(df["num_children"] > 3).sum()}')

train_set = replace_child_outliers(train_set, child_median)
val_set = replace_child_outliers(val_set, child_median)
test_set = replace_child_outliers(test_set, child_median)

**Insights**

- Nilai `num_children` di atas 3 (yaitu 9 dan 10, hanya 3 baris di `train_set`) dianggap kesalahan input dan diganti dengan median `train_set`.
- Batas 3 dipilih karena nilai 0–3 lebih sering muncul. 
- Outlier pada kolom lain, misalnya `nightly_rate_idr` dan `days_before_arrival`, tidak diubah karena nilainya masih masuk akal dan menjadi sinyal pembatalan.

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
INT_COLS = ['num_adults', 'num_children', 'special_requests', 'days_before_arrival', 'needs_parking']

for df in (train_set, val_set, test_set):
    df[INT_COLS] = df[INT_COLS].astype(int)

In [ ]:
for name, df in [('train', train_set), ('val', val_set), ('test', test_set)]:
    print(f'{name:5s} shape={df.shape}  missing={df.isna().sum().sum()}  num_children max={df["num_children"].max()}')

print('Baris identik setelah imputasi di train_set:', train_set.drop(columns=[ID_COL]).duplicated().sum())
train_set.dtypes

In [ ]:
def correct_guest_counts(df, adult_mode):
    df = df.copy()
    zero_guest = (df['num_adults'] == 0) & (df['num_children'] == 0)
    df.loc[zero_guest, 'num_adults'] = adult_mode
    return df

adult_mode = train_set['num_adults'].mode().iloc[0]

for name, df in [('train', train_set), ('val', val_set), ('test', test_set)]:
    zero_guest = (df['num_adults'] == 0) & (df['num_children'] == 0)
    print(f'{name} booking tidak valid karena ga ada orang: {zero_guest.sum()}')

train_set = correct_guest_counts(train_set, adult_mode)
val_set = correct_guest_counts(val_set, adult_mode)
test_set = correct_guest_counts(test_set, adult_mode)

In [ ]:
def clean_reservations(raw_fit, raw_eval, drop_duplicates=True):
    # drop_duplicates=False dipakai pipeline KNN kerabat yang membuang duplikat sendiri (agar baris tetap sejajar)
    train = raw_fit.drop_duplicates(subset=raw_fit.columns.drop(ID_COL)) if drop_duplicates else raw_fit
    fill = train[NUMERICAL_MISSING].median().to_dict()
    fill.update(train[CATEGORICAL_MISSING].mode().iloc[0].to_dict())
    child_median = train['num_children'].median()
    adult_mode = train.fillna(fill)['num_adults'].mode().iloc[0]

    frames = []
    for df in (train, raw_eval):
        df = replace_child_outliers(df.fillna(fill), child_median)
        df[INT_COLS] = df[INT_COLS].astype(int)
        frames.append(correct_guest_counts(df, adult_mode))
    train, other = frames
    return train, other

**Insights**

- Baris dengan 0 malam dan baris dengan harga 0 di luar channel Complimentary tidak diubah.
- Baris yang menjadi identik setelah imputasi tidak dihapus, karena kemiripannya berasal dari nilai hasil imputasi, bukan dari data asli.
- Ditemukan 1 booking tanpa orang (`num_adults` = 0 dan `num_children` = 0) pada `val_set`.

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

This part covers new attributes derived from existing ones.

In [ ]:
ROOM_MEDIAN_RATE = train_set.groupby('room_category')['nightly_rate_idr'].median()
RATE_MEDIAN      = train_set['nightly_rate_idr'].median()
LAST_MINUTE_DAYS = 7

def construct_attributes(df, room_rates=ROOM_MEDIAN_RATE, rate_median=RATE_MEDIAN):
    df = df.copy()
    if not CFG.FEATURE_ENGINEERING:
        return df

    df['has_children'] = (df['num_children'] > 0).astype(int)
    df['no_cost'] = (df['nightly_rate_idr'] == 0).astype(int)
    df['room_price_position'] = (df['nightly_rate_idr'] / df['room_category'].map(room_rates)).fillna(1.0)
    df['high_rate_no_request'] = ((df['nightly_rate_idr'] > rate_median) & (df['special_requests'] == 0)).astype(int)
    df['has_cancel'] = (df['prior_cancellations'] > 0).astype(int)
    
    arrival_date = pd.to_datetime(dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']))
    booking_date = arrival_date - pd.to_timedelta(df['days_before_arrival'], unit='D')
    
    df['week_day']         = arrival_date.dt.dayofweek
    df['booking_month']    = booking_date.dt.month
    df['booking_week_day'] = booking_date.dt.dayofweek
    df['online_last_minute'] = ((df['booking_channel'] == 'Online Travel Agent') & (df['days_before_arrival'] <= LAST_MINUTE_DAYS)).astype(int)
    df['no_special_requests'] = (df['special_requests'] == 0).astype(int)
    df['multiple_requests'] = (df['special_requests'] >= 2).astype(int)
    df['rate_no_requests'] = df['nightly_rate_idr'] * df['no_special_requests']
    df['online_no_requests'] = (df['booking_channel'] == 'Online Travel Agent').astype(int) * df['no_special_requests']

    lead_band = pd.cut(df['days_before_arrival'], [-1, 7, 30, 90, 180, np.inf]).astype(str)
    request_band = df['special_requests'].clip(upper=3).astype(str)
    df['channel_lead_band'] = df['booking_channel'] + '_' + lead_band
    df['channel_request_band'] = df['booking_channel'] + '_' + request_band
    df['arrival_month_category'] = df['arrival_month'].astype(str)

    return df

train_set = construct_attributes(train_set)
val_set   = construct_attributes(val_set)
test_set  = construct_attributes(test_set)

NEW_FEATURES = ['has_children', 'no_cost', 'room_price_position', 'high_rate_no_request', 'has_cancel',
                'week_day', 'booking_month', 'booking_week_day', 'online_last_minute',
                'no_special_requests', 'multiple_requests', 'rate_no_requests', 'online_no_requests',
                'channel_lead_band', 'channel_request_band', 'arrival_month_category']
NEW_FEATURES = [c for c in NEW_FEATURES if c in train_set.columns]
if CFG.FEATURE_ENGINEERING:
    display(train_set[NEW_FEATURES].describe().T)
else:
    print('Feature engineering dinonaktifkan.')

In [ ]:
if CFG.FEATURE_ENGINEERING:
    for c in ['has_children', 'no_cost', 'high_rate_no_request', 'has_cancel', 'online_last_minute', 'week_day', 'booking_month', 'booking_week_day']:
        print(train_set.groupby(c)[TARGET].agg(['mean', 'count']).round(3), '\n')

    print(train_set.groupby(pd.qcut(train_set['room_price_position'], 4, duplicates='drop'), observed=True)[TARGET].agg(['mean', 'count']).round(3))

**Insights**

- **`room_price_position`**: harga per malam dibagi median harga kategori kamar yang sama. 
- **`high_rate_no_request`**: booking dengan harga di atas median tanpa permintaan khusus. Tingkat pembatalannya 56,8%, jauh di atas rata-rata 33%.
- **`online_last_minute`**: booking melalui Online Travel Agent yang dibuat paling lambat 7 hari sebelum kedatangan. Tingkat pembatalannya hanya 11,1%.
- **`no_cost`** dan **`has_cancel`**: menandai menginap gratis (cancel 0,6%) dan tamu yang pernah membatalkan (cancel 4,4%).
- **`week_day`**, **`booking_month`**, dan **`booking_week_day`**: tanggal booking dihitung dari tanggal kedatangan - `days_before_arrival`.
- **`no_special_requests`**: menandai booking tanpa permintaan khusus.
- **`multiple_requests`**: menandai booking dengan minimal dua permintaan khusus.
- **`rate_no_requests`**: harga per malam untuk booking tanpa permintaan khusus, bernilai 0 jika ada permintaan khusus.
- **`online_no_requests`**: menandai booking melalui Online Travel Agent tanpa permintaan khusus.
- **`channel_lead_band`**: jalur pemesanan digabung dengan kelompok jeda kedatangan 0–7, 8–30, 31–90, 91–180, dan lebih dari 180 hari.
- **`channel_request_band`**: jalur pemesanan digabung dengan kelompok jumlah permintaan khusus 0, 1, 2, dan minimal 3.
- **`arrival_month_category`**: bulan kedatangan sebagai kategori untuk one-hot encoding.


### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
SCALES = NUMERICAL + DATE + ['room_price_position', 'week_day', 'booking_month', 'booking_week_day', 'rate_no_requests']
BINS = BINARY + ['has_children', 'no_cost', 'has_cancel', 'high_rate_no_request', 'online_last_minute', 'no_special_requests', 'multiple_requests', 'online_no_requests']
SCALES = [c for c in SCALES if c in train_set.columns]
BINS = [c for c in BINS if c in train_set.columns]

numeric_cols = [c for c in train_set.select_dtypes('number').columns if c != TARGET]
assert set(SCALES) | set(BINS) == set(numeric_cols) and not set(SCALES) & set(BINS)
assert train_set[BINS].isin([0, 1]).all().all()

def scale_features(df, scaler):
    df = df.copy()
    df[SCALES] = scaler.transform(df[SCALES])
    return df

scaler = StandardScaler()
scaler.fit(train_set[SCALES])

train_scaled = scale_features(train_set, scaler)
val_scaled = scale_features(val_set, scaler)
test_scaled = scale_features(test_set, scaler)

print('Di-scale     :', len(SCALES), 'kolom')
print('Tetap 0/1    :', BINS)
train_scaled[SCALES].describe().T[['mean', 'std', 'min', 'max']].round(2)

**Insight**

- Pakai standardisasi menggunakan z score, bukan min max normalization karena outlier dari dataset awal masih dipertahankan.
- Dilakukan scaling karena data terlalu bervariasi dan beragam. 
- Mean dan std hanya dihitung dari `train_set` saja, supaya hasil dari val_set dan test_set tidak bocor.

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
MIN_COUNT = 20

meal_map = {'Full Board': 'Half Board'}   
room_counts = train_set['room_category'].value_counts()
rare_rooms = room_counts[room_counts < MIN_COUNT].index.tolist()
print('Kamar yang sedikit:', rare_rooms, '| Full Board di train:', (train_set['meal_package'] == 'Full Board').sum())

def merging(df, rare_rooms=rare_rooms):
    df['meal_package'] = df['meal_package'].replace(meal_map)
    df['room_category'] = df['room_category'].replace({r: 'Type Other' for r in rare_rooms})
    return df

for df in (train_set, val_set, test_set, train_scaled, val_scaled, test_scaled):
    merging(df)

In [ ]:
MEALS = ['Room Only', 'Breakfast', 'Half Board']
ONEHOTRESULT = ['room_category', 'booking_channel']
if CFG.FEATURE_ENGINEERING:
    ONEHOTRESULT += ['channel_lead_band', 'channel_request_band', 'arrival_month_category']

meal_encoder = OrdinalEncoder(categories=[MEALS])
onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
meal_encoder.fit(train_scaled[['meal_package']])
onehot.fit(train_scaled[ONEHOTRESULT])

def onehotencode(df, meal_encoder=meal_encoder, onehot=onehot):
    df = df.copy()
    df['meal_package'] = meal_encoder.transform(df[['meal_package']])
    oh = pd.DataFrame(onehot.transform(df[ONEHOTRESULT]), columns=onehot.get_feature_names_out(ONEHOTRESULT), index=df.index)
    return pd.concat([df.drop(columns=ONEHOTRESULT), oh], axis=1)

train_a, val_a, test_a = onehotencode(train_scaled), onehotencode(val_scaled), onehotencode(test_scaled)

NB_EXCLUDED = ['no_special_requests', 'multiple_requests', 'rate_no_requests', 'online_no_requests', 'channel_lead_band', 'channel_request_band', 'arrival_month_category']
nb_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
nb_encoder.fit(train_set[CATEGORICAL])

def ordinalencode(df):
    df = df.drop(columns=NB_EXCLUDED, errors='ignore').copy()
    df[CATEGORICAL] = nb_encoder.transform(df[CATEGORICAL]).astype(int)
    return df

train_b, val_b, test_b = ordinalencode(train_set), ordinalencode(val_set), ordinalencode(test_set)

print('One Hot Encoding:', train_a.shape, val_a.shape, test_a.shape)
print('Ordinal Encoding:', train_b.shape, val_b.shape, test_b.shape)
print(onehot.get_feature_names_out(ONEHOTRESULT))

In [ ]:
def transform_features_a(train, other):
    room_rates = train.groupby('room_category')['nightly_rate_idr'].median()
    rate_median = train['nightly_rate_idr'].median()
    train = construct_attributes(train, room_rates, rate_median)
    other = construct_attributes(other, room_rates, rate_median)

    scaler = StandardScaler().fit(train[SCALES])
    counts = train['room_category'].value_counts()
    rare = counts[counts < MIN_COUNT].index
    train = merging(scale_features(train, scaler), rare)
    other = merging(scale_features(other, scaler), rare)

    meal = OrdinalEncoder(categories=[MEALS]).fit(train[['meal_package']])
    nominal = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(train[ONEHOTRESULT])
    train = onehotencode(train, meal, nominal)
    other = onehotencode(other, meal, nominal)
    return train, other

**Insights**

- `Full Board` (4 baris) dan beberapa kategori kamar (kurang dari 20 baris di `train_set`) terlalu sedikit untuk digunakan dalam training. `Full Board` digabung ke `Half Board` karena merupakan paket terdekat dalam urutan, sedangkan kategori kamar yang sedikit digabung menjadi `Type Other`.
- `room_category` dan `booking_channel` dilakukan one hot encoding karena kedua kolom tidak memiliki urutan. 
- `meal_package` dilakukan ordinal encoding karena memiliki urutan berdasarkan jumlah makanan yang termasuk (Room Only < Breakfast < Half Board), sehingga ordinal encoding mempertahankan informasi urutan dalam satu kolom.

### Data Discretization

In [ ]:
class AutoBinner:
    def __init__(self, n_bins=5, max_unique=40):
        self.n_bins, self.max_unique = n_bins, max_unique

    def fit(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        self.cols_ = [c for c in X.columns if X[c].nunique() > self.max_unique]
        self.bins_ = KBinsDiscretizer(n_bins=self.n_bins, encode='ordinal', strategy='quantile').fit(X[self.cols_])
        self.max_ = self._codes(X).max()              
        return self

    def _codes(self, X):
        X = pd.DataFrame(X).reset_index(drop=True).copy()
        X[self.cols_] = self.bins_.transform(X[self.cols_])
        return X.clip(lower=0).round().astype(int)

    def transform(self, X):
        return self._codes(X).clip(upper=self.max_, axis=1)


# Contoh pada data hasil encoding (train_b); pipeline Naive Bayes memakai kelas yang sama per fold
FEATURES_NB = [c for c in train_b.columns if c not in (ID_COL, TARGET)]
binner_demo = AutoBinner(n_bins=5, max_unique=40).fit(train_b[FEATURES_NB])
train_nb = binner_demo.transform(train_b[FEATURES_NB])
train_nb[TARGET] = train_b[TARGET].to_numpy()

print('Kolom yang dibagi ke bin:', binner_demo.cols_)
for c, edges in zip(binner_demo.cols_, binner_demo.bins_.bin_edges_):
    print(f'{c:22s}', np.round(edges, 2))

In [ ]:
FEATURES_NB = [c for c in train_nb.columns if c not in (ID_COL, TARGET)]
train_nb[FEATURES_NB].nunique().sort_values(ascending=False)

**Insights**

- Diskretisasi dibutuhkan oleh Naive Bayes, karena peluang dihitung dari frekuensi setiap nilai kategori.
- `AutoBinner` hanya membagi kolom dengan lebih dari 40 nilai unik (lead time, harga, posisi harga) ke dalam 5 bin *equal frequency*, sehingga setiap bin punya cukup data. Kolom lain sudah diskret dan dipakai apa adanya.
- Nilai di data evaluasi yang melebihi nilai terbesar di data fit dipotong ke nilai terbesar itu, sehingga tidak muncul kategori baru yang tidak punya frekuensi.
- Kelas yang sama dipakai oleh pipeline Naive Bayes (`prepare_nb_candidates`), dan batas bin dihitung ulang dari data fit di setiap fold.

### Feature Selection

In [ ]:
FEATURES_NB = [c for c in train_nb.columns if c not in (ID_COL, TARGET)]

rows = []
for c in FEATURES_NB:
    table = pd.crosstab(train_nb[c], train_nb[TARGET])            # contingency table fitur × target
    chi2_stat, p_value, dof, _ = chi2_contingency(table)
    mi = mutual_info_score(train_nb[c], train_nb[TARGET])
    rows.append({'feature': c, 'chi2': chi2_stat, 'dof': dof, 'p_value': p_value, 'mutual_info': mi})

fs_scores = pd.DataFrame(rows).sort_values('mutual_info', ascending=False).reset_index(drop=True)
fs_scores

In [ ]:
NOMINAL_COLS = ['room_category', 'booking_channel']
CORR_COLS = [c for c in FEATURES_NB if c not in NOMINAL_COLS]

corr = train_nb[CORR_COLS].corr(method='spearman')

plt.figure(figsize=(16, 13))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=True, fmt='.2f', annot_kws={'size': 7})
plt.title('Korelasi Spearman antar fitur (train_nb)')
plt.show()

In [ ]:
CORR_THRESHOLD = 0.8
upper = corr.abs().where(np.triu(np.ones(corr.shape, dtype=bool), k=1))  
strong_pairs = upper.stack().loc[lambda s: s >= CORR_THRESHOLD].sort_values(ascending=False)
strong_pairs

**Insights**

- Setelah diskretisasi, seluruh fitur dan target `is_canceled` bersifat kategorikal, sehingga uji yang sesuai adalah Chi-Squared pada *contingency table* dan Mutual Information.
- Korelasi Spearman dipakai karena data berupa nilai diskret; pasangan dengan korelasi ≥ 0,8 (misalnya `prior_cancellations` dan `has_cancel`) membawa informasi yang hampir sama.
- Tabel ini adalah analisis *filter* yang menilai setiap fitur sendiri-sendiri. Keputusan akhir seleksi fitur Naive Bayes diambil dengan *backward elimination* (metode *wrapper*) di Preparation Pipeline, karena metode itu langsung mengukur Macro F1 model dengan kombinasi fitur, termasuk efek fitur yang redundan.

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
FEATURES_A = [c for c in train_a.columns if c not in (ID_COL, TARGET)]
print('Jumlah fitur versi A:', len(FEATURES_A))

pca_full = PCA(random_state=CFG.SEED).fit(train_a[FEATURES_A])
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker='o')
plt.axhline(0.95, color='red', linestyle='--', label='95% variansi')
plt.xlabel('Jumlah komponen')
plt.ylabel('Variansi kumulatif yang dijelaskan')
plt.legend()
plt.show()

n_components = int(np.argmax(cum_var >= 0.95) + 1)
print(f'Komponen untuk 95% variansi: {n_components} dari {len(FEATURES_A)} fitur')

In [ ]:
pca = PCA(n_components=n_components, random_state=CFG.SEED).fit(train_a[FEATURES_A])

def apply_pca(df):
    pcs = pca.transform(df[FEATURES_A])
    out = pd.DataFrame(pcs, columns=[f'PC{i+1}' for i in range(n_components)], index=df.index)
    out.insert(0, ID_COL, df[ID_COL].values)
    if TARGET in df:
        out[TARGET] = df[TARGET].values
    return out

train_pca, val_pca, test_pca = apply_pca(train_a), apply_pca(val_a), apply_pca(test_a)
print(train_pca.shape, val_pca.shape, test_pca.shape)

**Insights**

- Beberapa fitur versi A saling berkorelasi (misalnya `prior_cancellations` dengan `has_cancel`, `num_children` dengan `has_children`, `nightly_rate_idr` dengan `room_price_position`). PCA merangkumnya menjadi komponen yang tidak saling berkorelasi.
- Jumlah komponen dipilih sebagai jumlah minimum yang mempertahankan 95% variansi: 28 dari 92 fitur, sehingga dimensi berkurang sekitar 70%.
- Representasi PCA dipakai SVM, karena waktu training SVM bertambah cepat seiring jumlah dimensi. Di pipeline, scaler dan PCA di-fit ulang dari data fit setiap fold (`prepare_features_pca`).

## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
print(train_set[TARGET].value_counts())
print(train_set[TARGET].value_counts(normalize=True).round(3))

In [ ]:
def oversample_index(y, seed=CFG.SEED):
    # Indeks baris setelah random oversampling: semua baris + salinan acak kelas minoritas sampai seimbang
    y = np.asarray(y)
    minority = int((y == 1).sum() < (y == 0).sum())
    pool = np.flatnonzero(y == minority)
    extra = np.random.default_rng(seed).choice(pool, size=len(y) - 2 * len(pool), replace=True)
    return np.concatenate([np.arange(len(y)), extra])

idx_bal = oversample_index(train_set[TARGET].to_numpy())
print('Data awal :', len(train_set), train_set[TARGET].value_counts(normalize=True).round(3).to_dict())
print('Data akhir:', len(idx_bal), train_set[TARGET].iloc[idx_bal].value_counts(normalize=True).round(3).to_dict())

**Insights**

- Distribusi target pada `train_set` adalah 67% tidak batal dan 33% batal.
- Random oversampling menyalin baris kelas minoritas secara acak sampai jumlahnya sama dengan kelas mayoritas. Fungsi ini hanya mengembalikan indeks, sehingga bisa dipakai untuk matriks apa pun.
- Oversampling diuji pada Naive Bayes (bagian Modeling). Model lain memakai distribusi asli dan menangani ketidakseimbangan lewat ambang prediksi atau `class_weight`, karena menyalin baris membuat model berbasis jarak dan pohon cenderung *overfit* pada salinan yang sama.

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
def to_xy(df, features, dtype=float):
    X = df[features].to_numpy(dtype=dtype)
    y = df[TARGET].to_numpy(dtype=int) if TARGET in df else None
    return X, y

FEATURES_PCA = [c for c in train_pca.columns if c.startswith('PC')]

# Versi A (Logistic Regression) dan PCA (SVM)
X_train_a, y_train_a     = to_xy(train_a, FEATURES_A)
X_val_a, _               = to_xy(val_a, FEATURES_A)
X_test_a, _              = to_xy(test_a, FEATURES_A)
X_train_pca, y_train_pca = to_xy(train_pca, FEATURES_PCA)
X_val_pca, _             = to_xy(val_pca, FEATURES_PCA)
X_test_pca, _            = to_xy(test_pca, FEATURES_PCA)
test_ids = test_set[ID_COL].to_numpy()

In [ ]:
for name, arrs in {'A': (X_train_a, X_val_a, X_test_a), 'PCA': (X_train_pca, X_val_pca, X_test_pca)}.items():
    print(f'{name:4s}', [a.shape for a in arrs], '| NaN:', sum(np.isnan(a).sum() for a in arrs))

assert len(y_val) == len(X_val_a) == len(X_val_pca)
assert len(test_ids) == len(X_test_a) == len(sample_sub)
assert np.array_equal(y_train_a, y_train_pca)

**Insights**

- Logistic Regression memakai 92 fitur versi A (ter-scale dan one-hot), sedangkan SVM memakai 28 komponen PCA dari fitur yang sama.
- KNN, Decision Tree, dan Naive Bayes memakai representasi masing-masing yang dibentuk langsung oleh pipeline di bagian berikut, karena kebutuhannya berbeda (jarak berbobot, kategori asli untuk *multi-way split*, dan kode diskret).

---

# Preparation Pipeline <a name="6"></a>

---

<!-- rubric v3: fitted steps, fitted_prep_steps -->

The pipeline combines the preparation steps so they run in one consistent order.

Setiap fungsi `prepare_features_*` menerima data mentah (`raw_fit` dan `raw_eval`), mempelajari semua statistik hanya dari `raw_fit`, lalu menerapkannya ke `raw_eval`. Fungsi yang sama dipakai untuk K-Fold, validasi, dan pelatihan final submission.

| Fungsi | Dipakai oleh | Isi |
|---|---|---|
| `prepare_features_a` | Logistic Regression | cleaning, fitur tim, scaling, one-hot |
| `prepare_features_pca` | SVM | fitur A lalu PCA |
| `prepare_features_dtl` | Decision Tree baseline | cleaning, fitur garis waktu, kategori dikodekan angka (tanpa one-hot), kolom numerik di-scale (versi D) |
| `prepare_features_knn` | KNN baseline | cleaning, fitur garis waktu, one-hot, standardisasi, bobot √MI |
| `prepare_features_knn_kerabat` | KNN final (`KNNKerabat`) | imputasi lead time dari kerabat, pipeline KNN, ditambah kode aturan beda dan nilai toleransi di kolom paling kanan |
| `prepare_features_dtl_final` | Decision Tree final (`DTL_Main`, `DTL_Alt`) | versi data DTL terbaik dari eksperimen di bagian Improvements (D, D1, D2, atau D3) |
| `prepare_features_nb` | Naive Bayes | cleaning, fitur tim, fitur garis waktu, label encoding, `AutoBinner`, fitur hasil *backward elimination* |

In [ ]:
def prepare_features_a(raw_fit, raw_eval):
    train, other = clean_reservations(raw_fit, raw_eval)
    train, other = transform_features_a(train, other)
    features = train.columns.drop([ID_COL, TARGET])
    X_train, y_train = to_xy(train, features)
    X_other, _ = to_xy(other, features)
    return X_train, y_train, X_other

In [ ]:
def prepare_features_pca(raw_fit, raw_eval):
    X_fit, y_fit, X_eval = prepare_features_a(raw_fit, raw_eval)
    fold_pca = PCA(n_components=n_components, random_state=CFG.SEED).fit(X_fit)
    return fold_pca.transform(X_fit), y_fit, fold_pca.transform(X_eval)

X_fit_check, _, X_val_check = prepare_features_pca(train_raw, val_raw)
print('PCA train sama:', np.allclose(X_fit_check, X_train_pca), '| val sama:', np.allclose(X_val_check, X_val_pca))

In [ ]:
TIMELINE_BASE = pd.Timestamp('2023-01-01')
NOMINAL_RAW   = ['meal_package', 'room_category', 'booking_channel']
NOMINAL_TEAM  = ['channel_lead_band', 'channel_request_band', 'arrival_month_category']

def add_timeline_features(df):
    df = df.copy()
    arrival = pd.to_datetime(dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']), errors='coerce')
    booking = arrival - pd.to_timedelta(df['days_before_arrival'], unit='D')
    df['arrival_ordinal']   = (arrival - TIMELINE_BASE).dt.days
    df['booking_ordinal']   = (booking - TIMELINE_BASE).dt.days
    df['arrival_dayofyear'] = arrival.dt.dayofyear
    return df

def clean_and_construct(raw_fit, raw_eval, team_features=False, drop_duplicates=True):
    train, other = clean_reservations(raw_fit, raw_eval, drop_duplicates=drop_duplicates)
    nominal = list(NOMINAL_RAW)
    if team_features:
        room_rates = train.groupby('room_category')['nightly_rate_idr'].median()
        rate_median = train['nightly_rate_idr'].median()
        train = construct_attributes(train, room_rates, rate_median)
        other = construct_attributes(other, room_rates, rate_median)
        nominal += [c for c in NOMINAL_TEAM if c in train.columns]
    train, other = add_timeline_features(train), add_timeline_features(other)
    counts = train['room_category'].value_counts()
    rare = counts[counts < MIN_COUNT].index
    train, other = merging(train.copy(), rare), merging(other.copy(), rare)
    return train, other, nominal

def onehot_frames(train, other, nominal):
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(train[nominal].astype(str))
    def encode(df):
        onehot = pd.DataFrame(encoder.transform(df[nominal].astype(str)), columns=encoder.get_feature_names_out(nominal), index=df.index)
        return pd.concat([df.drop(columns=nominal), onehot], axis=1)
    return encode(train), encode(other)

def prepare_features_base(raw_fit, raw_eval):
    train, other, nominal = clean_and_construct(raw_fit, raw_eval)
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(train[nominal].astype(str))
    def encode(df):
        onehot = pd.DataFrame(encoder.transform(df[nominal].astype(str)), columns=encoder.get_feature_names_out(nominal), index=df.index)
        return pd.concat([df.drop(columns=nominal), onehot], axis=1)
    train, other = encode(train), encode(other)
    features = [c for c in train.columns if c not in (ID_COL, TARGET)]
    X_fit, y_fit = to_xy(train, features)
    X_eval, _ = to_xy(other, features)
    return X_fit, y_fit, X_eval

def knn_feature_weights(X_fit, y_fit):
    binary = np.array([len(np.unique(X_fit[:, j])) <= 2 for j in range(X_fit.shape[1])])
    return np.sqrt(mutual_info_classif(X_fit, y_fit, discrete_features=binary, random_state=CFG.SEED))

def prepare_features_knn(raw_fit, raw_eval):
    X_fit, y_fit, X_eval = prepare_features_base(raw_fit, raw_eval)
    scaler = StandardScaler().fit(X_fit)
    weights = knn_feature_weights(X_fit, y_fit)
    return scaler.transform(X_fit) * weights, y_fit, scaler.transform(X_eval) * weights

X_base_check, y_base_check, X_base_val_check = prepare_features_base(train_raw, val_raw)
print('Base:', X_base_check.shape, X_base_val_check.shape, '| NaN:', np.isnan(X_base_check).sum() + np.isnan(X_base_val_check).sum())

**Pipeline KNN Kerabat: Aturan Beda dan Jarak**

Pada materi kNN, jarak atribut simbolik bernilai 0 jika sama dan 1 jika berbeda. `KNNKerabat` memakai ide ini sebagai **aturan beda**, masing-masing dengan penalti sendiri:

$$D(q, x) = \Big(\sum_j |q_j - x_j|^p + \sum_r \lambda_r \,\text{beda}_r(q, x)\Big)^{1/p}$$

- **Aturan simbolik** (kode harus sama): tanggal kedatangan, pasangan (malam weekend, malam weekday), pasangan (dewasa, anak), kanal, kamar, paket makan, permintaan khusus, parkir, dan riwayat tamu. Kode `-1` berarti nilai asli kosong, sehingga tidak diberi penalti (nilai hasil imputasi tidak dihukum dua kali).
- **Aturan toleransi** (selisih tidak boleh melebihi batas): lead time dengan tiga batas (7, 15, 30 hari) dan log harga dengan dua batas (3% dan 10%). Beberapa batas untuk satu kolom membentuk penalti bertingkat: makin jauh selisihnya, makin banyak aturan yang dilanggar.

Nilai $\lambda$ ditulis dalam satuan $s$, yaitu median jarak dasar (pangkat $p$) ke tetangga ke-10 pada data fit, sehingga skalanya tetap masuk akal di setiap fold.

Bagian pertama tetap pipeline KNN tim (one-hot, standardisasi, bobot √MI), ditambah imputasi kerabat dan transformasi log pada lead time dan harga.

In [ ]:
from algorithm.knn import KNN
from algorithm.knn_kerabat import KNNKerabat

ATURAN_SIMBOL = {
    'tanggal':    ['arrival_year', 'arrival_month', 'arrival_day'],
    'malam':      ['weekend_nights', 'weekday_nights'],
    'tamu':       ['num_adults', 'num_children'],
    'kanal':      ['booking_channel'],
    'kamar':      ['room_category'],
    'makan':      ['meal_package'],
    'permintaan': ['special_requests'],
    'parkir':     ['needs_parking'],
    'riwayat':    ['is_returning_guest', 'prior_cancellations', 'prior_completed_bookings'],
}
ATURAN_TOLERANSI = [('lead>7', 'days_before_arrival', 7.0), ('lead>15', 'days_before_arrival', 15.0),
                    ('lead>30', 'days_before_arrival', 30.0),
                    ('harga>3%', 'log_rate', np.log(1.03)), ('harga>10%', 'log_rate', np.log(1.10))]
NAMA_ATURAN = list(ATURAN_SIMBOL) + [nama for nama, _, _ in ATURAN_TOLERANSI]
TOLERANSI = np.array([batas for _, _, batas in ATURAN_TOLERANSI])
N_SIMBOL = len(ATURAN_SIMBOL)


def _kode_kolom(fit, ev, kolom):
    # Kode bilangan bulat dari kombinasi nilai di data fit. Kosong -> -1, nilai yang tidak ada di fit -> kode baru.
    def rapikan(df):
        sub = df[kolom].copy()
        for c in kolom:
            sub[c] = sub[c].astype(float) if pd.api.types.is_numeric_dtype(sub[c]) else sub[c].astype(object)
        return sub
    fit_sub, ev_sub = rapikan(fit), rapikan(ev)
    peta = fit_sub.dropna().drop_duplicates().reset_index(drop=True)
    peta['_kode'] = np.arange(len(peta))
    hasil = []
    for sub in (fit_sub, ev_sub):
        kode = np.array(sub.merge(peta, on=kolom, how='left')['_kode'], dtype=float)
        kode[np.isnan(kode)] = len(peta)
        kode[~sub.notna().all(axis=1).to_numpy()] = -1
        hasil.append(kode.astype(np.int64))
    return hasil


def kode_aturan(fit, ev):
    pasangan = [_kode_kolom(fit, ev, kolom) for kolom in ATURAN_SIMBOL.values()]
    S_fit = np.column_stack([a for a, _ in pasangan])
    S_eval = np.column_stack([b for _, b in pasangan])
    kontinu = lambda df: {'days_before_arrival': df['days_before_arrival'].to_numpy(dtype=float),
                          'log_rate': np.log1p(df['nightly_rate_idr'].to_numpy(dtype=float))}
    kf, ke = kontinu(fit), kontinu(ev)
    T_fit = np.column_stack([kf[kolom] for _, kolom, _ in ATURAN_TOLERANSI])
    T_eval = np.column_stack([ke[kolom] for _, kolom, _ in ATURAN_TOLERANSI])
    return S_fit, S_eval, T_fit, T_eval


PaketKNN = namedtuple('PaketKNN', 'X_fit y_fit X_eval y_eval S_fit S_eval T_fit T_eval features')


def siapkan_knn_kerabat(raw_fit, raw_eval, imputasi=True, skala_log=True, buang_duplikat=True):
    fit = raw_fit.drop_duplicates(subset=raw_fit.columns.drop(ID_COL)) if buang_duplikat else raw_fit
    fit, ev = fit.reset_index(drop=True), raw_eval.reset_index(drop=True)
    if imputasi:
        imputer = ImputasiLeadKerabat().fit(fit)
        fit, ev = imputer.transform(fit), imputer.transform(ev)
    S_fit, S_eval, T_fit, T_eval = kode_aturan(fit, ev)

    train, other, nominal = clean_and_construct(fit, ev, drop_duplicates=False)
    if skala_log:
        for c in ('days_before_arrival', 'nightly_rate_idr'):
            train[c], other[c] = np.log1p(train[c]), np.log1p(other[c])
    train, other = onehot_frames(train, other, nominal)
    features = [c for c in train.columns if c not in (ID_COL, TARGET)]
    X_fit, y_fit = to_xy(train, features)
    X_eval, y_eval = to_xy(other, features)
    assert len(X_fit) == len(S_fit) and len(X_eval) == len(S_eval)

    scaler = StandardScaler().fit(X_fit)
    weights = knn_feature_weights(X_fit, y_fit)
    return PaketKNN(scaler.transform(X_fit) * weights, y_fit, scaler.transform(X_eval) * weights, y_eval,
                    S_fit, S_eval, T_fit, T_eval, features)


paket_cek = siapkan_knn_kerabat(train_raw, val_raw)
print('X_fit:', paket_cek.X_fit.shape, '| X_eval:', paket_cek.X_eval.shape)
print('Aturan beda:', NAMA_ATURAN)
print('Persen kode kosong per aturan simbolik (fit):',
      dict(zip(ATURAN_SIMBOL, np.round(100 * (paket_cek.S_fit < 0).mean(axis=0), 1).tolist())))


def prepare_features_knn_kerabat(raw_fit, raw_eval):
    # Antarmuka sama dengan pipeline lain: X = [fitur numerik | kode simbol | nilai kontinu]
    paket = siapkan_knn_kerabat(raw_fit, raw_eval, **OPSI_KNN)
    X_fit = np.hstack([paket.X_fit, paket.S_fit, paket.T_fit])
    X_eval = np.hstack([paket.X_eval, paket.S_eval, paket.T_eval])
    return X_fit, paket.y_fit, X_eval

**Pipeline Decision Tree**

C4.5 bisa langsung memecah atribut kategorikal menjadi beberapa cabang (multi-way split), sehingga kategori cukup dikodekan menjadi angka dengan `OrdinalEncoder`, tanpa one-hot. Indeks kolom kategorikal diberikan ke parameter `categorical_features` agar tetap dipecah multi-way.

Kolom numerik tetap distandardisasi. Tujuannya bukan menyamakan skala (pohon tidak terpengaruh skala), tetapi karena implementasi C4.5 kelompok menganggap kolom bilangan bulat dengan sedikit nilai unik (misalnya `arrival_month` dan `special_requests`) sebagai kategorikal. Setelah di-scale, kolom tersebut dipecah dengan ambang (≤ t / > t) dan bisa dipakai berulang kali, yang lebih sesuai untuk atribut berurutan. Pilihan ini didasarkan pada eksperimen di bagian Decision Tree Improvement.

In [ ]:
def make_prepare_dtl_tree(team_features=False, numeric_as_continuous=True):
    # Mengembalikan fungsi prepare yang menghasilkan (X_fit, y_fit, X_eval, indeks kolom kategorikal, nama fitur)
    def prepare(raw_fit, raw_eval):
        train, other, nominal = clean_and_construct(raw_fit, raw_eval, team_features=team_features)
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit(train[nominal].astype(str))
        for df in (train, other):
            df[nominal] = encoder.transform(df[nominal].astype(str))
        features = [c for c in train.columns if c not in (ID_COL, TARGET)]
        if numeric_as_continuous:
            numeric = [c for c in features if c not in nominal]
            scaler = StandardScaler().fit(train[numeric])
            train[numeric], other[numeric] = scaler.transform(train[numeric]), scaler.transform(other[numeric])
        X_fit, y_fit = to_xy(train, features)
        X_eval, _ = to_xy(other, features)
        return X_fit, y_fit, X_eval, [features.index(c) for c in nominal], features
    return prepare

prepare_dtl_full = make_prepare_dtl_tree(team_features=False, numeric_as_continuous=True)

def prepare_features_dtl(raw_fit, raw_eval):
    # Antarmuka yang sama dengan pipeline lain: (X_fit, y_fit, X_eval)
    X_fit, y_fit, X_eval, _, _ = prepare_dtl_full(raw_fit, raw_eval)
    return X_fit, y_fit, X_eval

X_train_dtl, y_train_dtl, X_val_dtl, DTL_CAT, DTL_NAMES = prepare_dtl_full(train_raw, val_raw)
print('DTL:', X_train_dtl.shape, X_val_dtl.shape, '| NaN:', np.isnan(X_train_dtl).sum() + np.isnan(X_val_dtl).sum())
print('Kolom kategorikal:', [DTL_NAMES[i] for i in DTL_CAT])

**Pipeline Decision Tree Kerabat**

Pohon tidak menghitung jarak, jadi informasi kerabat diberikan sebagai fitur. Semua statistik dihitung dari data fit saja:
- `banyak_kerabat_*`: banyak pemesanan lain di data fit dengan kunci yang sama (untuk baris fit, dirinya sendiri tidak dihitung). Fitur ini berupa hitungan baris, tidak memakai label, dan bukan penjumlahan antarkolom.
- `rasio_batal_kerabat_*`: proporsi batal kerabat dengan *smoothing* $\frac{\text{batal} + m \cdot \bar{y}}{n + m}$, $m = 5$. Untuk baris fit, nilainya dihitung *out-of-fold* (5 lipatan) sehingga label baris itu sendiri tidak pernah ikut. Rata-rata *leave-one-out* sengaja tidak dipakai, karena nilainya bergeser sesuai label baris sendiri sehingga pohon bisa "mencontek".

Setiap fitur dibuat untuk kunci ketat (tanggal kedatangan, malam weekend, malam weekday, kanal) dan kunci longgar (tanggal kedatangan, kanal). Versi data yang dipakai dipilih di bagian Decision Tree Improvement.

In [ ]:
def encode_dtl_frames(train, other, nominal):
    # Isi make_prepare_dtl_tree (numeric_as_continuous=True): kategori dikodekan angka, kolom numerik di-scale
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit(train[nominal].astype(str))
    for df in (train, other):
        df[nominal] = encoder.transform(df[nominal].astype(str))
    features = [c for c in train.columns if c not in (ID_COL, TARGET)]
    numeric = [c for c in features if c not in nominal]
    scaler = StandardScaler().fit(train[numeric])
    train[numeric], other[numeric] = scaler.transform(train[numeric]), scaler.transform(other[numeric])
    X_fit, y_fit = to_xy(train, features)
    X_eval, _ = to_xy(other, features)
    return X_fit, y_fit, X_eval, [features.index(c) for c in nominal], features


PRIOR_M = 5
KUNCI_STATISTIK = {'ketat': KUNCI_KERABAT, 'longgar': KUNCI_LONGGAR}


def _rasio(banyak_batal, n, prior):
    return (banyak_batal + PRIOR_M * prior) / (n + PRIOR_M)


def tambah_statistik_kerabat(train, other, pakai_rasio=True):
    train, other = train.copy().reset_index(drop=True), other.copy().reset_index(drop=True)
    prior = train[TARGET].mean()
    for nama, kunci in KUNCI_STATISTIK.items():
        g = train.groupby(kunci)[TARGET].agg(['sum', 'count'])
        gabung = other[kunci].merge(g.reset_index(), on=kunci, how='left')
        n_other = gabung['count'].fillna(0).to_numpy()
        n_train = train.groupby(kunci)[TARGET].transform('count').to_numpy() - 1
        train[f'banyak_kerabat_{nama}'], other[f'banyak_kerabat_{nama}'] = n_train, n_other
        if pakai_rasio:
            other[f'rasio_batal_kerabat_{nama}'] = _rasio(gabung['sum'].fillna(0).to_numpy(), n_other, prior)
            oof = np.empty(len(train))
            for idx_a, idx_b in KFold(n_splits=5, shuffle=True, random_state=CFG.SEED).split(train):
                ga = train.iloc[idx_a].groupby(kunci)[TARGET].agg(['sum', 'count'])
                gb = train.iloc[idx_b][kunci].merge(ga.reset_index(), on=kunci, how='left')
                oof[idx_b] = _rasio(gb['sum'].fillna(0).to_numpy(), gb['count'].fillna(0).to_numpy(), prior)
            train[f'rasio_batal_kerabat_{nama}'] = oof
    return train, other


def buat_prepare_dtl_kerabat(imputasi=True, banyak=True, rasio=True):
    def prepare(raw_fit, raw_eval):
        fit, ev = raw_fit, raw_eval
        if imputasi:
            imputer = ImputasiLeadKerabat().fit(fit)
            fit, ev = imputer.transform(fit), imputer.transform(ev)
        train, other, nominal = clean_and_construct(fit, ev)
        if banyak:
            train, other = tambah_statistik_kerabat(train, other, pakai_rasio=rasio)
        return encode_dtl_frames(train, other, nominal)
    return prepare


VERSI_DATA_DTL = {
    'D: kategori asli + garis waktu':           prepare_dtl_full,
    'D1: + imputasi kerabat':                   buat_prepare_dtl_kerabat(imputasi=True, banyak=False, rasio=False),
    'D2: + imputasi + banyak kerabat':          buat_prepare_dtl_kerabat(imputasi=True, banyak=True, rasio=False),
    'D3: + imputasi + banyak & rasio kerabat':  buat_prepare_dtl_kerabat(imputasi=True, banyak=True, rasio=True),
}

In [ ]:
def prepare_nb_candidates(raw_fit, raw_eval):
    train, other, nominal = clean_and_construct(raw_fit, raw_eval, team_features=True)
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit(train[nominal].astype(str))
    for df in (train, other):
        df[nominal] = encoder.transform(df[nominal].astype(str)).astype(int)
    features = [c for c in train.columns if c not in (ID_COL, TARGET)]
    binner = AutoBinner(n_bins=5, max_unique=40).fit(train[features])
    return binner.transform(train[features]), train[TARGET].to_numpy(dtype=int), binner.transform(other[features])

def prepare_features_nb(raw_fit, raw_eval, features=None):
    X_fit, y_fit, X_eval = prepare_nb_candidates(raw_fit, raw_eval)
    features = NB_SELECTED if features is None else features
    return X_fit[features].to_numpy(dtype=int), y_fit, X_eval[features].to_numpy(dtype=int)

In [ ]:
from algorithm.naive import NaiveBayes

nb_select_folds = []
for fit_idx, eval_idx in StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.SEED).split(train_raw, train_raw[TARGET]):
    X_fit, y_fit, X_eval = prepare_nb_candidates(train_raw.iloc[fit_idx], train_raw.iloc[eval_idx])
    nb_select_folds.append((X_fit, y_fit, X_eval, train_raw.iloc[eval_idx][TARGET].to_numpy()))

def nb_cv_score(columns):
    scores = []
    for X_fit, y_fit, X_eval, y_eval in nb_select_folds:
        pred = NaiveBayes(alpha=1.0).fit(X_fit[columns], y_fit).predict(X_eval[columns])
        scores.append(f1_score(y_eval, pred, average='macro'))
    return np.mean(scores)

MIN_GAIN = 0.0005
NB_CANDIDATES = list(nb_select_folds[0][0].columns)
NB_SELECTED = list(NB_CANDIDATES)
nb_selection_score = nb_cv_score(NB_SELECTED)
nb_selection_log = [{'Fitur dibuang': '-', 'Jumlah fitur': len(NB_SELECTED), 'Macro F1 K-Fold': nb_selection_score}]

while len(NB_SELECTED) > 1:
    trials = {c: nb_cv_score([x for x in NB_SELECTED if x != c]) for c in NB_SELECTED}
    drop, score = max(trials.items(), key=lambda kv: kv[1])
    if score <= nb_selection_score + MIN_GAIN:
        break
    NB_SELECTED.remove(drop)
    nb_selection_score = score
    nb_selection_log.append({'Fitur dibuang': drop, 'Jumlah fitur': len(NB_SELECTED), 'Macro F1 K-Fold': score})

print(f'Terpilih {len(NB_SELECTED)} dari {len(NB_CANDIDATES)} kandidat fitur')
pd.DataFrame(nb_selection_log).round(4)

**Insights**

- Kandidat fitur Naive Bayes adalah fitur asli, fitur buatan tim, dan fitur garis waktu. Kategori cukup dikodekan dengan label encoding, karena Naive Bayes menghitung frekuensi per nilai sehingga tidak perlu one-hot.
- Kolom dengan lebih dari 40 nilai unik dibagi ke 5 bin *equal frequency* oleh `AutoBinner`.
- Seleksi fitur memakai *backward elimination* (metode *wrapper*) dengan Macro F1 5-Fold pada `train_raw`: fitur dibuang satu per satu selama Macro F1 naik lebih dari 0,0005. Fitur yang tersisa (`NB_SELECTED`) dipakai oleh `prepare_features_nb`. Fitur yang redundan menurut analisis korelasi biasanya ikut terbuang di sini, karena Naive Bayes menghitung informasi yang sama dua kali.

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

<!-- rubric v3: pairwise distances, val_neighbor_dists -->

This part covers the nearest neighbor classifier and its validation.

In [ ]:
from algorithm.knn import KNN

X_train_knn, y_train_knn, X_val_knn = prepare_features_knn(train_raw, val_raw)
X_train_base, _, X_val_base = prepare_features_base(train_raw, val_raw)
base_scaler = StandardScaler().fit(X_train_base)
X_train_scaled, X_val_scaled = base_scaler.transform(X_train_base), base_scaler.transform(X_val_base)

knn_variants = {
    'Euclidean (p=2), tanpa bobot fitur': (KNN(k=15, p=2), X_train_scaled, X_val_scaled),
    'Manhattan (p=1), tanpa bobot fitur': (KNN(k=15, p=1), X_train_scaled, X_val_scaled),
    'Manhattan (p=1), bobot sqrt(MI)':    (KNN(k=15, p=1), X_train_knn, X_val_knn),
}
knn_results = []
for name, (model, X_fit, X_eval) in knn_variants.items():
    pred = model.fit(X_fit, y_train_knn).predict(X_eval)
    knn_results.append({'Variasi (k = 15)': name, 'Macro F1 validasi': f1_score(y_val, pred, average='macro')})

knn_model = knn_variants['Manhattan (p=1), bobot sqrt(MI)'][0]
y_pred_knn = knn_model.predict(X_val_knn)
pd.DataFrame(knn_results).set_index('Variasi (k = 15)').round(4)

In [ ]:
print(classification_report(y_val, y_pred_knn, digits=4))
print(confusion_matrix(y_val, y_pred_knn))

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
from algorithm.DTL_main import DTL_Main

dtl_baseline_params = dict(max_depth=12, min_samples_split=20, min_samples_leaf=5)
dtl_model = DTL_Main(**dtl_baseline_params, categorical_features=DTL_CAT)
dtl_model.fit(X_train_dtl, y_train_dtl, feature_names=DTL_NAMES)
dtl_pred = dtl_model.predict(X_val_dtl)

print(f'Macro F1 validasi DTL baseline: {f1_score(y_val, dtl_pred, zero_division=0, average="macro"):.4f}')
print(classification_report(y_val, dtl_pred, digits=4))

dtl_model.plot_tree(top_n=5, save_path=CFG.OUTPUT_DIR / 'dtl_tree_baseline.png')
plt.show()

### Decision Tree Alternative (`DTL_Alt`)

Variasi eksperimental C4.5 (`DTL_Alt`) yang menambahkan dukungan *cost-sensitive learning* (`class_weight`), *threshold tuning*, MSD-Splitting 4-bin (`continuous_split='msd'`), dan aproksimasi L'Hospital (`criterion='lhospital'`) di atas `DTL_Main`.

In [ ]:
from algorithm.DTL_alt import DTL_Alt

dtl_alt_baseline_params = dict(
    max_depth=14,
    min_samples_split=20,
    min_samples_leaf=5,
    class_weight='balanced',
    continuous_split='binary',
    criterion='gain_ratio',
)
dtl_alt_model = DTL_Alt(**dtl_alt_baseline_params, categorical_features=DTL_CAT)
dtl_alt_model.fit(X_train_dtl, y_train_dtl, feature_names=DTL_NAMES)
dtl_alt_pred = dtl_alt_model.predict(X_val_dtl)

print(f'Macro F1 validasi DTL_Alt baseline: {f1_score(y_val, dtl_alt_pred, zero_division=0, average="macro"):.4f}')
print(classification_report(y_val, dtl_alt_pred, digits=4))

dtl_alt_model.plot_tree(top_n=5, save_path=CFG.OUTPUT_DIR / 'dtl_alt_tree_baseline.png')
plt.show()

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.


Eksperimen memakai fitur A dan ambang 0,5. Ketiga penalti ikut dievaluasi pada holdout dan K-Fold.


Pada eksperimen awal ini, semua penalti memakai lambda = 0,01. Pemilihan konfigurasi berdasarkan K-Fold dilakukan pada bagian Improvements.

In [ ]:
from algorithm.log_regression import LogisticRegression

lr_options = dict(learning_rate=0.001, max_iteration=100, random=CFG.SEED)
lr_models = {}
lr_predictions = {}
lr_results = []

for penalty in ('l1', 'l2', 'elasticnet'):
    model = LogisticRegression(penalty=penalty, lmda=0.01, **lr_options)
    model.fit(X_train_a, y_train_a)
    predictions = model.predict(X_val_a)
    lr_models[penalty] = model
    lr_predictions[penalty] = predictions
    lr_results.append({'Penalti': penalty, 'Macro F1 validasi': f1_score(y_val, predictions, zero_division=0, average='macro')})

pd.DataFrame(lr_results).set_index('Penalti').round(4)

### Bonus: Visualisasi Lintasan Training Logistic Regression

Visualisasi berikut menggunakan satu fitur untuk menampilkan lintasan parameter `θ₀` (bias) dan `θ₁` (bobot fitur) pada contour log-loss. 

In [ ]:
from algorithm.log_regression_visualization import save_loss_video
from IPython.display import Video, display

feature_name = "days_before_arrival"
X_video = train_a[[feature_name]].to_numpy(dtype=float)
y_video = y_train_a

try:
    video_path = save_loss_video(
        X_video,
        y_video,
        path=CFG.OUTPUT_DIR / "log_regression_loss.mp4",
        feature_name=feature_name,
    )
    display(Video(str(video_path), embed=True))
except RuntimeError as error:
    print('Video tidak dibuat:', error)

## Support Vector Machine

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
from algorithm.SVM import Svm
import time

X_svm_fit, y_svm_fit = X_train_pca, y_train_pca
X_svm_val, y_svm_val = X_val_pca, y_val

svm_models = {}
svm_predictions = {}
svm_results = []
svm_max_samples = 500
for name, kernel, extra in (
    ('Linear', 'linear', {}),
    ('Polynomial', 'polynomial', {'degree': 2, 'coef0': 0.7}),
    ('Gaussian RBF', 'rbf', {'gamma': 0.05}),
):
    model = Svm(
        kernel=kernel,
        C=1.0,
        n_iters=250,
        tolerance=1e-5,
        multi_class='ovo',
        max_samples=svm_max_samples,
        random_state=CFG.SEED,
        **extra,
    )
    svm_start = time.perf_counter()
    model.fit(X_svm_fit, y_svm_fit)
    svm_training_time = time.perf_counter() - svm_start
    pred = model.predict(X_svm_val)
    svm_models[name] = model
    svm_predictions[name] = pred
    svm_results.append({
        'Kernel': name,
        'Jumlah data latih': min(len(X_svm_fit), svm_max_samples),
        'Support vector': len(model.support_vectors),
        'Macro F1 validasi': f1_score(y_svm_val, pred, average='macro'),
        'Konvergen': model.converged_,
        'Iterasi': model.n_iterations_,
        'Waktu latih (detik)': round(svm_training_time, 4),
    })

pd.DataFrame(svm_results).set_index('Kernel').round(4)

### Bonus: SVM Optimization using Adaboost
Penggunaan algoritma SVM sebagai base estimator dengan metode AdaBoost untuk meningkatkan performa klasifikasi melalui pembobotan sampel secara iteratif dan penggabungan beberapa model SVM menggunakan weighted voting.

AdaBoost adalah *ensemble*, sehingga bagian ini hanya bonus: modelnya tidak ikut pemilihan model final dan tidak dipakai untuk submission (syarat bonus Kaggle adalah tanpa ensemble).

In [ ]:
from algorithm.AdaBoostSVM import AdaBoostSvm
import time

X_svm_fit, y_svm_fit = X_train_pca, y_train_pca
X_svm_val, y_svm_val = X_val_pca, y_val

adaboost_models = {}
adaboost_predictions = {}
adaboost_results = []

svm_max_samples = 500
n_estimators = 5

for name, kernel, extra in (
    ('Linear', 'linear', {}),
    ('Polynomial', 'polynomial', {'degree': 2, 'coef0': 0.7}),
    ('Gaussian RBF', 'rbf', {'gamma': 0.05}),
):
    model = AdaBoostSvm(
        n_estimators=n_estimators,
        learning_rate=0.5,
        kernel=kernel,
        C=1.0,
        n_iters=250,
        tolerance=1e-5,
        multi_class='ovo',
        max_samples=svm_max_samples,
        random_state=CFG.SEED,
        **extra,
    )

    start_time = time.perf_counter()
    model.fit(X_svm_fit, y_svm_fit)
    training_time = time.perf_counter() - start_time

    pred = model.predict(X_svm_val)

    adaboost_models[name] = model
    adaboost_predictions[name] = pred

    adaboost_results.append({
        'Kernel': name,
        'Jumlah data latih': len(X_svm_fit),
        'Jumlah estimator': model.n_estimators,
        'Macro F1 validasi': f1_score(
            y_svm_val, pred, average='macro'
        ),
        'Accuracy validasi': accuracy_score(y_svm_val, pred),
        'Konvergen': model.converged_,
        'Early stopping': model.stopped_early_,
        'Waktu latih (detik)': round(training_time, 4),
    })

pd.DataFrame(adaboost_results).set_index('Kernel').round(4)

### BONUS : Bidang Pemisah, Margin, dan Support Vector

Visualisasi memakai PCA 2 komponen dan sampel berlapis agar interpretasi kernel dapat dilihat dalam bidang 2D. Garis kontur 0 adalah bidang pemisah; kontur -1 dan +1 menandai batas margin.

In [ ]:
Svm.plot_decision_boundaries(
    X_svm_fit,
    y_svm_fit,
    save_dir=CFG.OUTPUT_DIR / 'images',
    max_samples=700,
    seed=CFG.SEED,
    models={
        **{f'SVM {name}': model for name, model in svm_models.items()},
        **{f'AdaBoostSVM {name}': model for name, model in adaboost_models.items()},
    },
)

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
from algorithm.naive import NaiveBayes

X_train_nb, y_train_nb, X_val_nb = prepare_features_nb(train_raw, val_raw)

idx_bal_nb = oversample_index(y_train_nb)              # fungsi dari bagian Class Balance
X_train_nb_bal, y_train_nb_bal = X_train_nb[idx_bal_nb], y_train_nb[idx_bal_nb]

nb_models = {}
nb_predictions = {}
nb_results = []

for name, (X_tr, y_tr) in {'asli': (X_train_nb, y_train_nb), 'oversampled': (X_train_nb_bal, y_train_nb_bal)}.items():
    model = NaiveBayes(alpha=1.0).fit(X_tr, y_tr)
    predictions = model.predict(X_val_nb)
    nb_models[name] = model
    nb_predictions[name] = predictions
    nb_results.append({'Data latih': name, 'Macro F1 validasi': f1_score(y_val, predictions, average='macro')})

pd.DataFrame(nb_results).set_index('Data latih').round(4)

In [ ]:
nb_model = nb_models['asli']
y_pred_nb = nb_predictions['asli']

print(classification_report(y_val, y_pred_nb, digits=4))
print(confusion_matrix(y_val, y_pred_nb))

## Model Evaluation

### Holdout

Tabel berikut adalah titik awal: setiap model dengan konfigurasi awal dari bagian Modeling, dievaluasi pada `val_raw`. Konfigurasi final setiap model ditentukan dengan K-Fold di bagian Improvements, lalu dirangkum di "Pemilihan Model Terbaik".

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

models = {
    'KNN': (knn_model, prepare_features_knn, y_pred_knn),
    'Decision Tree': (dtl_model, prepare_features_dtl, dtl_pred),
    'Decision Tree (Alt)': (dtl_alt_model, prepare_features_dtl, dtl_alt_pred),
    'Logistic Regression (L1)': (lr_models['l1'], prepare_features_a, lr_predictions['l1']),
    'Logistic Regression (L2)': (lr_models['l2'], prepare_features_a, lr_predictions['l2']),
    'Logistic Regression (Elastic Net)': (lr_models['elasticnet'], prepare_features_a, lr_predictions['elasticnet']),
    'SVM (Linear)': (svm_models['Linear'], prepare_features_pca, svm_predictions['Linear']),
    'SVM (Polynomial)': (svm_models['Polynomial'], prepare_features_pca, svm_predictions['Polynomial']),
    'SVM (Gaussian RBF)': (svm_models['Gaussian RBF'], prepare_features_pca, svm_predictions['Gaussian RBF']),
    'Naive Bayes': (nb_model, prepare_features_nb, y_pred_nb),   
}
active_models = {name: entry for name, entry in models.items() if entry is not None}

In [ ]:
holdout = {}
for name in active_models:
    model, prepare, pred = active_models[name]
    holdout[name] = {
        'Accuracy': accuracy_score(y_val, pred),
        'Precision': precision_score(y_val, pred, zero_division=0, average='macro'),
        'Recall': recall_score(y_val, pred, zero_division=0, average='macro'),
        'Macro F1': f1_score(y_val, pred, zero_division=0, average='macro'),
    }
holdout_scores = pd.DataFrame(holdout).T
holdout_scores.round(4)

### Precision, recall, dan F1 per model

Grafik berikut membandingkan ketiga skor pada data holdout untuk setiap model yang sudah terhubung.

In [ ]:
holdout_scores[['Precision', 'Recall', 'Macro F1']].T.plot.bar(
    rot=0, ylim=(0, 1), ylabel='Skor holdout', colormap='tab10',
)
plt.tight_layout()
plt.show()

**Insights**

- *(Isi setelah dijalankan: model mana yang paling kuat dengan konfigurasi awal, dan model mana yang paling jauh di bawah.)*
- Semua model memakai ambang 0,5 dan parameter awal di sini, sehingga skornya belum mencerminkan performa final. Model yang menghasilkan peluang (KNN kerabat, DTL_Alt, Logistic Regression) disetel ambangnya di bagian Improvements.

### K-Fold Cross Validation

Semua tuning di bagian Improvements memakai 3-Fold stratified yang sama (`splits`, dibuat tepat setelah split data). Pada setiap fold, preprocessing dan model dilatih ulang hanya dari bagian latih fold tersebut. K-Fold untuk konfigurasi awal tidak dihitung terpisah, karena skor yang dipakai untuk memilih model adalah skor K-Fold konfigurasi hasil tuning.

## Improvements (Optional)

### Logistic Regression Improvement

Kode berikut mencoba beberapa kombinasi penalti, lambda, dan ambang prediksi pada fold yang sama. Rata-rata F1 setiap kombinasi ditampilkan dalam tabel untuk membandingkan pengaruh parameternya.

In [ ]:
thresholds = np.round(np.arange(0.25, 0.651, 0.025), 3)
lambdas = [0.001, 0.01]
cv_rows = []

for fit_idx, eval_idx in splits:
    fold_train = train_raw.iloc[fit_idx]
    fold_eval = train_raw.iloc[eval_idx]
    X_fit, y_fit, X_eval = prepare_features_a(fold_train, fold_eval)
    y_eval = fold_eval[TARGET]

    for penalty in lr_models:
        for lmda in lambdas:
            fold_lr = LogisticRegression(penalty=penalty, lmda=lmda, **lr_options)
            fold_lr.fit(X_fit, y_fit)
            prob = fold_lr.probability(X_eval)

            for threshold in thresholds:
                pred = prob >= threshold
                score = f1_score(y_eval, pred, zero_division=0, average='macro')
                cv_rows.append({'Penalti': penalty, 'Lambda': lmda, 'Ambang': threshold, 'Macro F1': score})
cv_results = pd.DataFrame(cv_rows)
cv_summary = cv_results.groupby(['Penalti', 'Lambda', 'Ambang'])['Macro F1'].mean()
cv_summary.unstack('Ambang').round(4)

### K-Nearest Neighbors Improvement: KNN Kerabat

Bagian ini memakai `KNNKerabat` (`src/algorithm/knn_kerabat.py`, turunan `KNN` kelompok) dengan pipeline `prepare_features_knn_kerabat`, lalu mencari versi data, metrik jarak ($p$), besar penalti setiap aturan beda, `k`, bobot suara (q = 0 berarti voting mayoritas biasa seperti `KNN`), dan threshold dengan 3-Fold CV yang sama (`splits`). Ini adalah *hyperparameter tuning* dan modifikasi fungsi jarak (*heterogeneous distance*, Wilson & Martinez, 1997) serta voting berbobot jarak (Dudani, 1976), bukan algoritma optimasi.

#### Mesin Tuning yang Cepat

Tuning KNN biasanya lambat karena setiap konfigurasi menghitung ulang jarak ke seluruh data latih. Di sini:
- Jarak dasar (perkalian matriks untuk $p = 2$) dan bit pelanggaran aturan dihitung **sekali per fold**. Setiap konfigurasi penalti hanya mengambil total penalti dari *lookup table* berukuran $2^{14}$ lalu memilih tetangga dengan `argpartition`.
- Tetangga disimpan sampai `K_MAX`, sehingga semua `k`, bobot suara, dan threshold dinilai dari hasil yang sama dengan jumlah kumulatif.
- Macro F1 untuk ratusan kombinasi dihitung sekaligus dengan operasi matriks.

Skor setiap konfigurasi adalah rata-rata Macro F1 3-Fold, dengan `k`, bobot suara, dan threshold terbaiknya.

In [ ]:
K_LIST = np.array([5, 9, 15, 21, 25, 31, 41, 51])
Q_LIST = np.array([0.0, 1.0, 1.5, 2.0, 3.0, 4.0])          # bobot suara 1 / d^q (0 = mayoritas biasa)
THR_LIST = np.round(np.arange(0.30, 0.701, 0.02), 2)
K_MAX = int(K_LIST.max())


def macro_f1_banyak(y, pred):
    # pred: (n, m) bool -> Macro F1 untuk setiap kolom
    y = y.astype(float)
    pred = pred.astype(np.float32)
    tp1 = y @ pred
    n_pred1 = pred.sum(axis=0)
    fp1, fn1 = n_pred1 - tp1, y.sum() - tp1
    tp0 = len(y) - tp1 - fp1 - fn1
    f1_1 = 2 * tp1 / np.maximum(2 * tp1 + fp1 + fn1, 1)
    f1_0 = 2 * tp0 / np.maximum(2 * tp0 + fn1 + fp1, 1)
    return (f1_0 + f1_1) / 2


def skor_tetangga(jarak, idx, y_fit, y_eval):
    # Macro F1 untuk semua (bobot suara, k, threshold): array (len(Q), len(K), len(THR))
    label = y_fit[idx]
    skor = np.empty((len(Q_LIST), len(K_LIST), len(THR_LIST)))
    for iq, q in enumerate(Q_LIST):
        bobot = np.ones_like(jarak) if q == 0 else 1.0 / (jarak + 1e-9) ** q
        peluang = np.cumsum(bobot * label, axis=1)[:, K_LIST - 1] / np.cumsum(bobot, axis=1)[:, K_LIST - 1]
        pred = peluang[:, :, None] >= THR_LIST[None, None, :]
        skor[iq] = macro_f1_banyak(y_eval, pred.reshape(len(y_eval), -1)).reshape(len(K_LIST), len(THR_LIST))
    return skor


def nilai_konfigurasi(paket_folds, p, daftar_lambda):
    # daftar_lambda: list vektor penalti dalam satuan s. Hasil: (n_konfigurasi, Q, K, THR), rata-rata fold
    skor = np.zeros((len(daftar_lambda), len(Q_LIST), len(K_LIST), len(THR_LIST)))
    for paket in paket_folds:
        model = KNNKerabat(k=K_MAX, p=p, toleransi=TOLERANSI, random_state=CFG.SEED)
        model.fit(paket.X_fit, paket.y_fit, paket.S_fit, paket.T_fit)
        s = model.skala_jarak()          # median jarak dasar ke tetangga ke-10 (satuan penalti)
        hasil = model.kneighbors(paket.X_eval, paket.S_eval, paket.T_eval,
                                 daftar_penalti=[np.asarray(lam) * s for lam in daftar_lambda])
        for i, (jarak, idx) in enumerate(hasil):
            skor[i] += skor_tetangga(jarak, idx, paket.y_fit, paket.y_eval) / len(paket_folds)
    return skor


def ringkas(skor, nama):
    baris = []
    for i, n in enumerate(nama):
        iq, ik, it = np.unravel_index(np.argmax(skor[i]), skor[i].shape)
        baris.append({'Konfigurasi': n, 'Macro F1 K-Fold': skor[i].max(), 'k': int(K_LIST[ik]),
                      'Bobot suara q': float(Q_LIST[iq]), 'Threshold': float(THR_LIST[it])})
    return pd.DataFrame(baris).sort_values('Macro F1 K-Fold', ascending=False, ignore_index=True)


def lambda_seragam(nilai_simbol, toleransi_aktif, nilai_toleransi=None):
    # Penalti sama untuk semua aturan simbolik; aturan toleransi yang aktif diberi nilai_toleransi
    lam = np.zeros(len(NAMA_ATURAN))
    lam[:N_SIMBOL] = nilai_simbol
    for nama in toleransi_aktif:
        lam[NAMA_ATURAN.index(nama)] = nilai_simbol if nilai_toleransi is None else nilai_toleransi
    return lam


def bangun_fold(**opsi):
    return [siapkan_knn_kerabat(train_raw.iloc[f], train_raw.iloc[e], **opsi) for f, e in splits]

#### Tahap A: Versi Data KNN

Empat versi data dibandingkan, masing-masing dengan dan tanpa aturan beda (penalti seragam $1s$, toleransi lead 15 hari dan harga 10%), memakai $p = 2$.

In [ ]:
t0 = time.time()
VERSI_DATA_KNN = {
    'K1: pipeline tim + log':                 dict(imputasi=False, skala_log=True,  buang_duplikat=True),
    'K2: + imputasi kerabat':                 dict(imputasi=True,  skala_log=True,  buang_duplikat=True),
    'K3: + imputasi kerabat, tanpa log':      dict(imputasi=True,  skala_log=False, buang_duplikat=True),
    'K4: + imputasi kerabat, duplikat tetap': dict(imputasi=True,  skala_log=True,  buang_duplikat=False),
}
lam_default = lambda_seragam(1.0, ['lead>15', 'harga>10%'])
lam_nol = np.zeros(len(NAMA_ATURAN))

baris_a, fold_cache = [], {}
for nama, opsi in VERSI_DATA_KNN.items():
    fold_cache[nama] = bangun_fold(**opsi)
    skor = nilai_konfigurasi(fold_cache[nama], p=2, daftar_lambda=[lam_nol, lam_default])
    for label, s in zip(['tanpa aturan beda', 'dengan aturan beda'], skor):
        baris_a.append({'Data': nama, 'Aturan': label, **ringkas(s[None], [''])
                        .drop(columns='Konfigurasi').iloc[0].to_dict()})

tahap_a = pd.DataFrame(baris_a).sort_values('Macro F1 K-Fold', ascending=False, ignore_index=True)
print(f'Waktu tahap A: {(time.time() - t0) / 60:.1f} menit')
tahap_a.round(4)

In [ ]:
# Versi data terbaik (berdasarkan skor dengan aturan beda) dipakai untuk tahap berikutnya.
# Untuk memaksa versi tertentu, isi DATA_KNN dengan salah satu nama di VERSI_DATA_KNN.
DATA_KNN = None
if DATA_KNN is None:
    dengan_aturan = tahap_a[tahap_a['Aturan'] == 'dengan aturan beda']
    DATA_KNN = dengan_aturan[dengan_aturan['Data'].isin(VERSI_DATA_KNN)].iloc[0]['Data']
OPSI_KNN = VERSI_DATA_KNN[DATA_KNN]
fold_knn = fold_cache[DATA_KNN]
print('Versi data KNN dipakai:', DATA_KNN, OPSI_KNN)

**Insights**

- Aturan beda menaikkan skor pada setiap versi data (+0,007 sampai +0,012). Versi terbaik K3 (imputasi kerabat, tanpa log) mencapai 0,8480, dibanding 0,8414 tanpa aturan beda.
- Imputasi kerabat paling berguna jika dipasangkan dengan aturan beda (K3 0,8480 vs K1 0,8436), karena lead time hasil imputasi ikut menentukan siapa tetangga terdekat.
- Transformasi log tidak membantu, dan menyimpan duplikat juga tidak (0,8437 vs 0,8440).

#### Tahap B: Metrik Jarak dan Besar Penalti

Pada versi data terbaik, dicoba $p \in \{1, 2\}$, penalti seragam $\lambda \in \{0{,}25; 0{,}5; 1; 2; 4\}\,s$, dan tiga pola toleransi: tanpa toleransi, satu batas (lead 15 hari + harga 10%), dan bertingkat (semua batas lead dan harga, masing-masing setengah penalti).

In [ ]:
t0 = time.time()
POLA_TOLERANSI = {
    'tanpa toleransi': ([], 1.0),
    'satu batas': (['lead>15', 'harga>10%'], 1.0),
    'bertingkat': (['lead>7', 'lead>15', 'lead>30', 'harga>3%', 'harga>10%'], 0.5),
}
konfig_b, nama_b = [lam_nol], ['tanpa aturan beda']
for nilai in (0.25, 0.5, 1.0, 2.0, 4.0):
    for pola, (aktif, faktor) in POLA_TOLERANSI.items():
        konfig_b.append(lambda_seragam(nilai, aktif, nilai * faktor))
        nama_b.append(f'lambda={nilai:g}s, {pola}')

hasil_b = {}
for p in (1, 2):
    skor = nilai_konfigurasi(fold_knn, p=p, daftar_lambda=konfig_b)
    hasil_b[p] = ringkas(skor, nama_b).assign(p=p)
    print(f'p = {p}: {(time.time() - t0) / 60:.1f} menit')

tahap_b = pd.concat(hasil_b.values(), ignore_index=True).sort_values('Macro F1 K-Fold', ascending=False, ignore_index=True)
P_KNN = int(tahap_b.iloc[0]['p'])
lam_terbaik = konfig_b[nama_b.index(tahap_b.iloc[0]['Konfigurasi'])]
print('Terbaik tahap B:', tahap_b.iloc[0]['Konfigurasi'], '| p =', P_KNN)
tahap_b.head(12).round(4)

#### Tahap C dan D: Penalti per Aturan

Mulai dari konfigurasi terbaik tahap B, setiap aturan diubah sendiri-sendiri (dikali 0, 0,5, atau 2; aturan yang masih 0 dicoba 0,5 dan 1 kali penalti seragam terbaik). Semua 40-an konfigurasi ini dinilai dari satu kali perhitungan jarak per fold. Tahap D lalu menggabungkan perubahan yang membantu secara bertahap (*greedy*), mulai dari yang kenaikannya paling besar.

In [ ]:
t0 = time.time()
satuan = lam_terbaik[:N_SIMBOL].max()
konfig_c, nama_c, ubah_c = [lam_terbaik], ['awal (terbaik tahap B)'], [None]
for r, nama in enumerate(NAMA_ATURAN):
    pilihan = [0.0, 0.5 * lam_terbaik[r], 2.0 * lam_terbaik[r]] if lam_terbaik[r] > 0 else [0.5 * satuan, satuan]
    for nilai in pilihan:
        lam = lam_terbaik.copy()
        lam[r] = nilai
        konfig_c.append(lam)
        nama_c.append(f'{nama} = {nilai:g}s')
        ubah_c.append((r, nilai))

skor_c = nilai_konfigurasi(fold_knn, p=P_KNN, daftar_lambda=konfig_c)
tahap_c = ringkas(skor_c, nama_c)
skor_awal = skor_c[0].max()
print(f'Waktu tahap C: {(time.time() - t0) / 60:.1f} menit | skor awal {skor_awal:.4f}')
tahap_c.head(12).round(4)

In [ ]:
t0 = time.time()
MIN_NAIK = 0.0003
naik = {}
for i in range(1, len(konfig_c)):
    r, nilai = ubah_c[i]
    gain = skor_c[i].max() - skor_awal
    if gain > MIN_NAIK and gain > naik.get(r, (None, 0))[1]:
        naik[r] = (nilai, gain)          # perubahan terbaik untuk setiap aturan

urutan = sorted(naik.items(), key=lambda kv: -kv[1][1])
konfig_d, nama_d = [lam_terbaik], ['awal (terbaik tahap B)']
lam = lam_terbaik.copy()
for langkah, (r, (nilai, _)) in enumerate(urutan, start=1):
    lam = lam.copy()
    lam[r] = nilai
    konfig_d.append(lam)
    nama_d.append(f'langkah {langkah}: + {NAMA_ATURAN[r]} = {nilai:g}s (kumulatif)')

skor_d = nilai_konfigurasi(fold_knn, p=P_KNN, daftar_lambda=konfig_d) if len(konfig_d) > 1 else skor_c[:1]
tahap_d = ringkas(skor_d, nama_d[:len(skor_d)])

terbaik_d = tahap_d.iloc[0]
LAMBDA_KNN = konfig_d[nama_d.index(terbaik_d['Konfigurasi'])]
K_KNN, Q_KNN, THR_KNN = int(terbaik_d['k']), float(terbaik_d['Bobot suara q']), float(terbaik_d['Threshold'])
CV_KNN = float(terbaik_d['Macro F1 K-Fold'])

print(f'Waktu tahap D: {(time.time() - t0) / 60:.1f} menit')
print(f'Konfigurasi KNN final: p={P_KNN}, k={K_KNN}, q={Q_KNN}, threshold={THR_KNN}, Macro F1 K-Fold={CV_KNN:.4f}')
print(pd.Series(LAMBDA_KNN, index=NAMA_ATURAN, name='penalti (satuan s)').round(3).to_string())
tahap_d.round(4)

**Insights**

- Dengan aturan beda, $p = 2$ lebih baik daripada $p = 1$. Penalti seragam $1s$ dengan toleransi bertingkat memberi 0,8486.
- Tahap C dan D hanya menambah +0,0013 (penalti kanal 2s dan lead > 30 hari 1s), menjadi 0,8499 dengan k = 25, bobot suara q = 4, dan threshold 0,48. Kenaikan ini masih di dalam variasi antarfold, jadi penalti seragam sudah hampir optimal dan hasilnya tidak bergantung pada tuning yang terlalu halus.
- Banyak konfigurasi dipilih dengan CV yang sama, sehingga skor K-Fold terbaik sedikit optimis. `val_raw`, yang tidak dipakai saat tuning, menjadi pemeriksaan akhir di bawah.

#### Evaluasi Holdout KNN Kerabat

Model dilatih pada seluruh `train_raw` dengan versi data dan konfigurasi final, lalu diuji pada `val_raw`. Pembandingnya adalah KNN baseline dari bagian Modeling.

In [ ]:
def buat_knn_kerabat():
    return KNNKerabat(k=K_KNN, p=P_KNN, bobot_suara=Q_KNN, threshold=THR_KNN, toleransi=TOLERANSI,
                      penalti_relatif=LAMBDA_KNN, n_simbol=N_SIMBOL, n_kontinu=len(TOLERANSI), random_state=CFG.SEED)

t0 = time.time()
X_train_knn_kerabat, y_train_knn_kerabat, X_val_knn_kerabat = prepare_features_knn_kerabat(train_raw, val_raw)
tuned_knn_kerabat = buat_knn_kerabat().fit(X_train_knn_kerabat, y_train_knn_kerabat)
tuned_pred_knn_kerabat = tuned_knn_kerabat.predict(X_val_knn_kerabat)
waktu_knn_kerabat = time.time() - t0

holdout_knn = pd.DataFrame([
    {'Model': 'KNN baseline (k=15, Manhattan, bobot √MI)', 'Macro F1 holdout': f1_score(y_val, y_pred_knn, average='macro')},
    {'Model': 'KNN kerabat', 'Macro F1 holdout': f1_score(y_val, tuned_pred_knn_kerabat, average='macro')},
]).set_index('Model')
print(f'Waktu prepare + fit + predict KNN kerabat: {waktu_knn_kerabat:.1f} detik')
print(classification_report(y_val, tuned_pred_knn_kerabat, digits=4))
print(confusion_matrix(y_val, tuned_pred_knn_kerabat))
holdout_knn.round(4)

**Insights**

- Pada holdout, KNN kerabat mencapai Macro F1 0,8605, naik +0,033 dari KNN baseline (0,8274).
- Skor holdout lebih tinggi daripada K-Fold (0,8499) karena saat holdout model dilatih pada 21,9 ribu baris, bukan 14,6 ribu, sehingga lebih banyak kerabat yang ditemukan. Model submission dilatih pada 27,4 ribu baris.

### Decision Tree: Versi Data

Sebelum tuning, versi data untuk pohon dibandingkan dengan 3-Fold CV yang sama. Modelnya `DTL_Alt` dengan parameter awal dari bagian Modeling (`dtl_alt_baseline_params`) dan ambang disapu 0,35–0,60, sehingga yang berubah hanya datanya.

| Versi | Isi |
|---|---|
| A | `prepare_features_a`: fitur tim, scaling, one-hot (92 fitur) |
| D | kategori asli (dikodekan angka, dipecah *multi-way*) + fitur garis waktu, kolom numerik di-scale (`prepare_features_dtl`) |
| D1 | D + imputasi lead time dari kerabat |
| D2 | D1 + banyak kerabat (kunci ketat dan longgar) |
| D3 | D2 + rasio batal kerabat (*out-of-fold*) |

Versi terbaik menjadi `prepare_features_dtl_final`, dan data fold-nya langsung dipakai ulang oleh tuning `DTL_Main` dan `DTL_Alt` di bawah.

Kolom numerik di-scale karena implementasi C4.5 kelompok menganggap kolom bilangan bulat dengan sedikit nilai unik (misalnya `arrival_month`) sebagai kategorikal. Setelah di-scale, kolom itu dipecah dengan ambang (≤ t / > t) dan bisa dipakai berulang kali di satu jalur, yang lebih sesuai untuk atribut berurutan.

In [ ]:
import time
from sklearn.model_selection import ParameterGrid
from algorithm.DTL_main import DTL_Main
from algorithm.DTL_alt import DTL_Alt

def prepare_dtl_a(raw_fit, raw_eval):
    X_fit, y_fit, X_eval = prepare_features_a(raw_fit, raw_eval)
    return X_fit, y_fit, X_eval, None, [f'fitur_a_{j}' for j in range(X_fit.shape[1])]

dtl_alt_thresholds = np.round(np.arange(0.35, 0.601, 0.025), 3)
VERSI_DTL_UJI = {'A: fitur tim + one-hot + scaling': prepare_dtl_a, **VERSI_DATA_DTL}

t0 = time.time()
dtl_fold_cache, baris_dtl = {}, []
for nama, prepare in VERSI_DTL_UJI.items():
    t_versi = time.time()
    dtl_fold_cache[nama] = []
    skor = {th: [] for th in dtl_alt_thresholds}
    for fit_idx, eval_idx in splits:
        X_fit, y_fit, X_eval, cat, names = prepare(train_raw.iloc[fit_idx], train_raw.iloc[eval_idx])
        y_eval = train_raw.iloc[eval_idx][TARGET].to_numpy(dtype=int)
        dtl_fold_cache[nama].append((X_fit, y_fit, X_eval, y_eval, cat))
        prob = DTL_Alt(**dtl_alt_baseline_params, categorical_features=cat).fit(X_fit, y_fit).predict_proba(X_eval)[:, 1]
        for th in dtl_alt_thresholds:
            skor[th].append(f1_score(y_eval, (prob >= th).astype(int), average='macro'))
    th_best = max(skor, key=lambda t: np.mean(skor[t]))
    baris_dtl.append({'Data': nama, 'Jumlah fitur': X_fit.shape[1], 'Ambang': float(th_best),
                      'Macro F1 K-Fold': np.mean(skor[th_best]), 'Std K-Fold': np.std(skor[th_best]),
                      'Waktu 3-Fold (detik)': time.time() - t_versi})

tahap_dtl = pd.DataFrame(baris_dtl).sort_values('Macro F1 K-Fold', ascending=False, ignore_index=True)
print(f'Waktu eksperimen versi data DTL: {(time.time() - t0) / 60:.1f} menit')
tahap_dtl.round(4)

In [ ]:
# Versi data dengan Macro F1 K-Fold tertinggi dipakai untuk semua tuning DTL dan model final.
# Untuk memaksa versi tertentu, isi DATA_DTL dengan salah satu nama di VERSI_DTL_UJI.
DATA_DTL = None
if DATA_DTL is None:
    DATA_DTL = tahap_dtl.iloc[0]['Data']
prepare_dtl_final = VERSI_DTL_UJI[DATA_DTL]

def prepare_features_dtl_final(raw_fit, raw_eval):
    # Antarmuka sama dengan pipeline lain: (X_fit, y_fit, X_eval)
    X_fit, y_fit, X_eval, _, _ = prepare_dtl_final(raw_fit, raw_eval)
    return X_fit, y_fit, X_eval

X_train_dtl_final, y_train_dtl_final, X_val_dtl_final, DTL_CAT_FINAL, DTL_NAMES_FINAL = prepare_dtl_final(train_raw, val_raw)
dtl_fold_data = [(X_fit, y_fit, X_eval, y_eval) for X_fit, y_fit, X_eval, y_eval, _ in dtl_fold_cache[DATA_DTL]]
print('Versi data DTL dipakai:', DATA_DTL, f'({X_train_dtl_final.shape[1]} fitur)')
print('Kolom kategorikal:', [DTL_NAMES_FINAL[i] for i in DTL_CAT_FINAL] if DTL_CAT_FINAL else '-')

**Insights**

- *(Isi setelah dijalankan: versi data terbaik dan selisihnya terhadap versi D.)*
- Pada eksperimen sebelumnya (dengan parameter hasil tuning), D2 terbaik: K-Fold 0,8330 vs 0,8306 untuk D, dan kenaikannya hampir seluruhnya berasal dari imputasi kerabat (D1 = D2 = 0,8330). Rasio batal kerabat (D3) justru menurunkan skor, karena hanya sekitar setengah kelompok kerabat berlabel seragam sehingga rasionya berisik.
- Versi A memakai 92 fitur, sehingga sekitar dua kali lebih lambat daripada versi D tanpa menambah skor.

### Decision Tree (`DTL_Main`) Improvement

*Hyperparameter tuning* `max_depth`, `min_samples_leaf`, dan *post-pruning* (`None`, `'pep'`, `'rep'`) pada versi data terpilih, memakai data fold yang sudah disiapkan oleh eksperimen versi data (tanpa memproses ulang).

In [ ]:
dtl_grid = {
    'max_depth': [12, 18, 24],
    'min_samples_split': [20],
    'min_samples_leaf': [1, 5, 10],
    'pruning': [None, 'pep', 'rep'],
}

t0 = time.time()
dtl_cv_rows = []
for params in ParameterGrid(dtl_grid):
    if params['min_samples_split'] < 2 * params['min_samples_leaf']:
        continue
    for X_fit, y_fit, X_eval, y_eval in dtl_fold_data:
        pred = DTL_Main(**params, categorical_features=DTL_CAT_FINAL).fit(X_fit, y_fit).predict(X_eval)
        dtl_cv_rows.append({**params, 'Macro F1': f1_score(y_eval, pred, zero_division=0, average='macro')})

dtl_tuning = (
    pd.DataFrame(dtl_cv_rows)
    .groupby(['max_depth', 'min_samples_split', 'min_samples_leaf', 'pruning'], dropna=False)['Macro F1']
    .agg(['mean', 'std']).reset_index()
    .sort_values('mean', ascending=False, ignore_index=True)
)
dtl_tuning.columns = ['max_depth', 'min_samples_split', 'min_samples_leaf', 'pruning', 'Macro F1 K-Fold', 'Std K-Fold']
print(f'Waktu tuning DTL_Main: {(time.time() - t0) / 60:.1f} menit')
dtl_tuning.head(10).round(4)

In [ ]:
best_dtl_row = dtl_tuning.iloc[0]
best_dtl_params = {
    'max_depth': int(best_dtl_row['max_depth']),
    'min_samples_split': int(best_dtl_row['min_samples_split']),
    'min_samples_leaf': int(best_dtl_row['min_samples_leaf']),
    'pruning': None if pd.isna(best_dtl_row['pruning']) else str(best_dtl_row['pruning']),
}
best_dtl_cv_f1 = float(best_dtl_row['Macro F1 K-Fold'])

tuned_dtl = DTL_Main(**best_dtl_params, categorical_features=DTL_CAT_FINAL)
tuned_dtl.fit(X_train_dtl_final, y_train_dtl_final, feature_names=DTL_NAMES_FINAL)
tuned_pred_dtl = tuned_dtl.predict(X_val_dtl_final)
val_prob_dtl = tuned_dtl.predict_proba(X_val_dtl_final)[:, 1]

print('Parameter DTL_Main terpilih berdasarkan K-Fold:', best_dtl_params)
print(f'Macro F1 K-Fold  : {best_dtl_cv_f1:.4f}')
print(f'Macro F1 Holdout : {f1_score(y_val, tuned_pred_dtl, zero_division=0, average="macro"):.4f}')

tuned_dtl.plot_tree(top_n=5, save_path=CFG.OUTPUT_DIR / 'dtl_tree_tuned.png')
plt.show()

### Decision Tree Alternative (`DTL_Alt`) Improvement

*Hyperparameter tuning* untuk `DTL_Alt` dilakukan dalam dua tahap menggunakan `dtl_fold_data` (versi data terpilih, 3-Fold CV yang sama, bebas *leakage*):
1. **Tahap 1 — Eksplorasi Strategi Optimasi & Ambang Prediksi:** Menguji seluruh kombinasi `class_weight` (`None`, `'balanced'`), `continuous_split` (`'binary'`, `'msd'`), `criterion` (`'gain_ratio'`, `'lhospital'`), dan `pruning` (`None`, `'pep'`, `'rep'`), sekaligus menyapu ambang `threshold` (`0.35`–`0.60`) tanpa melatih ulang pohon.
2. **Tahap 2 — Penyetelan Struktur Pohon (`max_depth`, `min_samples_split`, `min_samples_leaf`) & Ambang Terbaik:** Memperhalus kedalaman (14, 18, 24), `min_samples_split` (10, 20), dan ukuran *leaf* (1, 5) pada kombinasi strategi terbaik dari Tahap 1. Hasil tahap ini adalah `DTL_Alt` final.

In [ ]:
# Tahap 1: Uji kombinasi 4 optimasi DTL_Alt (class_weight, continuous_split, criterion, pruning) + threshold tuning
dtl_alt_strategy_grid = {
    'class_weight': [None, 'balanced'],
    'continuous_split': ['binary', 'msd'],
    'criterion': ['gain_ratio', 'lhospital'],
    'pruning': [None, 'pep', 'rep'],
}
dtl_alt_thresholds = np.round(np.arange(0.35, 0.601, 0.025), 3)

base_struct = {
    'max_depth': best_dtl_params['max_depth'],
    'min_samples_split': best_dtl_params['min_samples_split'],
    'min_samples_leaf': best_dtl_params['min_samples_leaf'],
}

dtl_alt_stage1_rows = []
for strat in ParameterGrid(dtl_alt_strategy_grid):
    t0 = time.time()
    th_scores = {th: [] for th in dtl_alt_thresholds}

    for X_fit, y_fit, X_eval, y_eval in dtl_fold_data:
        fold_alt = DTL_Alt(**base_struct, **strat, categorical_features=DTL_CAT_FINAL)
        fold_alt.fit(X_fit, y_fit)
        prob_eval = fold_alt.predict_proba(X_eval)[:, 1]
        for th in dtl_alt_thresholds:
            pred_th = (prob_eval >= th).astype(int)
            th_scores[th].append(f1_score(y_eval, pred_th, zero_division=0, average='macro'))

    elapsed = time.time() - t0
    best_th = max(th_scores, key=lambda t: np.mean(th_scores[t]))
    dtl_alt_stage1_rows.append({
        **strat,
        'F1 K-Fold (th=0.5)': float(np.mean(th_scores[0.5])),
        'Ambang Terbaik': float(best_th),
        'Macro F1 K-Fold': float(np.mean(th_scores[best_th])),
        'Std K-Fold': float(np.std(th_scores[best_th])),
        'Waktu 3-Fold (s)': elapsed,
    })

dtl_alt_stage1 = pd.DataFrame(dtl_alt_stage1_rows).sort_values('Macro F1 K-Fold', ascending=False, ignore_index=True)
dtl_alt_stage1.round(4)

In [ ]:
# Tahap 2: Tuning struktur pohon (max_depth, min_samples_split, min_samples_leaf) + threshold pada strategi terbaik Tahap 1
best_strat_row = dtl_alt_stage1.iloc[0]
best_alt_strategy = {
    'class_weight': None if pd.isna(best_strat_row['class_weight']) else str(best_strat_row['class_weight']),
    'continuous_split': str(best_strat_row['continuous_split']),
    'criterion': str(best_strat_row['criterion']),
    'pruning': None if pd.isna(best_strat_row['pruning']) else str(best_strat_row['pruning']),
}
print('Strategi DTL_Alt terbaik dari Tahap 1:', best_alt_strategy)

dtl_alt_struct_grid = {
    'max_depth': [14, 18, 24],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [1, 5],
}

dtl_alt_stage2_rows = []
for struct_params in ParameterGrid(dtl_alt_struct_grid):
    if struct_params['min_samples_split'] < 2 * struct_params['min_samples_leaf']:
        continue
    th_scores = {th: [] for th in dtl_alt_thresholds}
    for X_fit, y_fit, X_eval, y_eval in dtl_fold_data:
        fold_alt = DTL_Alt(**struct_params, **best_alt_strategy, categorical_features=DTL_CAT_FINAL)
        fold_alt.fit(X_fit, y_fit)
        prob_eval = fold_alt.predict_proba(X_eval)[:, 1]
        for th in dtl_alt_thresholds:
            pred_th = (prob_eval >= th).astype(int)
            th_scores[th].append(f1_score(y_eval, pred_th, zero_division=0, average='macro'))

    best_th = max(th_scores, key=lambda t: np.mean(th_scores[t]))
    dtl_alt_stage2_rows.append({
        **struct_params,
        'Ambang': float(best_th),
        'Macro F1 K-Fold': float(np.mean(th_scores[best_th])),
        'Std K-Fold': float(np.std(th_scores[best_th])),
    })

dtl_alt_tuning = pd.DataFrame(dtl_alt_stage2_rows).sort_values('Macro F1 K-Fold', ascending=False, ignore_index=True)
dtl_alt_tuning.head(15).round(4)

In [ ]:
best_alt_row = dtl_alt_tuning.iloc[0]
best_dtl_alt_params = {
    'max_depth': int(best_alt_row['max_depth']),
    'min_samples_split': int(best_alt_row['min_samples_split']),
    'min_samples_leaf': int(best_alt_row['min_samples_leaf']),
    **best_alt_strategy,
}
best_dtl_alt_threshold = float(best_alt_row['Ambang'])
best_dtl_alt_cv_f1 = float(best_alt_row['Macro F1 K-Fold'])

tuned_dtl_alt = DTL_Alt(**best_dtl_alt_params, categorical_features=DTL_CAT_FINAL)
tuned_dtl_alt.fit(X_train_dtl_final, y_train_dtl_final, feature_names=DTL_NAMES_FINAL)
val_prob_dtl_alt = tuned_dtl_alt.predict_proba(X_val_dtl_final)[:, 1]
tuned_pred_dtl_alt = tuned_dtl_alt.predict(X_val_dtl_final, threshold=best_dtl_alt_threshold)

print('Parameter DTL_Alt terpilih berdasarkan K-Fold:', best_dtl_alt_params)
print(f'Ambang terbaik   : {best_dtl_alt_threshold:.3f}')
print(f'Macro F1 K-Fold  : {best_dtl_alt_cv_f1:.4f}')
print(f'Macro F1 Holdout : {f1_score(y_val, tuned_pred_dtl_alt, zero_division=0, average="macro"):.4f}')

tuned_dtl_alt.plot_tree(top_n=5, save_path=CFG.OUTPUT_DIR / 'dtl_alt_tree_tuned.png')
plt.show()

**Insights**

- *(Isi setelah dijalankan: strategi `DTL_Alt` terbaik dari tahap 1 (class weight, split kontinu, kriteria, pruning), struktur pohon terbaik dari tahap 2, dan perbandingannya dengan `DTL_Main`.)*
- Pada run sebelumnya, kombinasi terbaik adalah tanpa class weight, split biner, gain ratio, dan PEP, dengan depth 18 dan leaf 1 (holdout 0,8504 dengan data D2).

### Naive Bayes Improvement

Sesuai spesifikasi, Naive Bayes memakai frekuensi, Laplace smoothing +1, dan aturan MAP, sehingga `alpha` dan ambang tidak disetel. Improvement Naive Bayes ada di data: seleksi fitur dengan *backward elimination* (Preparation Pipeline) dan uji oversampling (Modeling). Sel berikut menghitung Macro F1 K-Fold model final pada fold yang sama dengan model lain, dibandingkan dengan memakai semua kandidat fitur.

In [ ]:
nb_feature_sets = {'Terpilih (backward elimination)': NB_SELECTED, 'Semua kandidat': NB_CANDIDATES}
nb_feature_rows = []
for label, feats in nb_feature_sets.items():
    scores = []
    for fit_idx, eval_idx in splits:
        fold_eval = train_raw.iloc[eval_idx]
        X_fit, y_fit, X_eval = prepare_features_nb(train_raw.iloc[fit_idx], fold_eval, features=feats)
        pred = NaiveBayes(alpha=1.0).fit(X_fit, y_fit).predict(X_eval)
        scores.append(f1_score(fold_eval[TARGET], pred, zero_division=0, average='macro'))
    nb_feature_rows.append({'Set fitur': label, 'Jumlah fitur': len(feats),
                            'Macro F1 K-Fold': np.mean(scores), 'Std K-Fold': np.std(scores)})

nb_feature_cv = pd.DataFrame(nb_feature_rows).set_index('Set fitur')
nb_cv_f1 = float(nb_feature_cv.loc['Terpilih (backward elimination)', 'Macro F1 K-Fold'])
nb_feature_cv.round(4)

**Insights**

- *(Isi setelah dijalankan: selisih Macro F1 K-Fold antara fitur hasil backward elimination dan semua kandidat, serta hasil oversampling di bagian Modeling.)*

### SVM Improvement dan Tuning

Tuning memakai 3-Fold yang sama dengan model lain. Data PCA disiapkan ulang per fold dengan `prepare_features_pca` (scaler dan PCA hanya di-fit dari bagian latih fold), sehingga tidak ada *leakage* dari fold evaluasi. Grid dibuat kecil karena SVM dilatih dengan solver QP (SLSQP) pada sampel 500 baris: `C` untuk linear, `C` dan degree untuk polynomial, serta `C` dan gamma untuk RBF. Iterasi dan toleransi sama dengan model final (`n_iters=250`, `tolerance=1e-5`).

In [ ]:
svm_grid = {
    'Linear': [{'C': C} for C in (0.1, 1.0, 10.0)],
    'Polynomial': [{'C': C, 'degree': degree, 'coef0': 1.0, 'gamma': 0.1} for C in (0.1, 1.0) for degree in (2, 3)],
    'Gaussian RBF': [{'C': C, 'gamma': gamma} for C in (1.0, 10.0) for gamma in (0.01, 0.05, 0.1)],
}
SVM_ITER = dict(n_iters=250, tolerance=1e-5)
SVM_KERNEL = {'Linear': 'linear', 'Polynomial': 'polynomial', 'Gaussian RBF': 'rbf'}

t0 = time.time()
svm_fold_data = []
for fit_idx, eval_idx in splits:
    X_fit, y_fit, X_eval = prepare_features_pca(train_raw.iloc[fit_idx], train_raw.iloc[eval_idx])
    svm_fold_data.append((X_fit, y_fit, X_eval, train_raw.iloc[eval_idx][TARGET].to_numpy(dtype=int)))

svm_tuning_rows = []
for kernel_name, options_list in svm_grid.items():
    for options in options_list:
        fold_scores = []
        for X_fit, y_fit, X_eval, y_eval in svm_fold_data:
            model = Svm(kernel=SVM_KERNEL[kernel_name], multi_class='ovo', max_samples=svm_max_samples,
                        random_state=CFG.SEED, **SVM_ITER, **options).fit(X_fit, y_fit)
            fold_scores.append(f1_score(y_eval, model.predict(X_eval), average='macro'))
        svm_tuning_rows.append({'Kernel': kernel_name, **options,
                                'Macro F1 K-Fold': np.mean(fold_scores), 'Std': np.std(fold_scores)})

svm_tuning = pd.DataFrame(svm_tuning_rows).sort_values('Macro F1 K-Fold', ascending=False, ignore_index=True)
print(f'Waktu tuning SVM: {(time.time() - t0) / 60:.1f} menit')
svm_tuning.round(4)

In [ ]:
best_svm_row = svm_tuning.iloc[0]
best_svm_kernel = SVM_KERNEL[best_svm_row['Kernel']]
best_svm_options = {'C': float(best_svm_row['C']), **SVM_ITER}
for key in ('degree', 'coef0', 'gamma'):
    if key in svm_tuning.columns and pd.notna(best_svm_row.get(key)):
        best_svm_options[key] = int(best_svm_row[key]) if key == 'degree' else float(best_svm_row[key])

tuned_svm = Svm(kernel=best_svm_kernel, multi_class='ovo', max_samples=svm_max_samples, random_state=CFG.SEED, **best_svm_options).fit(X_svm_fit, y_svm_fit)
tuned_pred_svm = tuned_svm.predict(X_svm_val)
print('SVM terbaik:', best_svm_row['Kernel'], best_svm_options)
print('Macro F1 holdout:', round(f1_score(y_svm_val, tuned_pred_svm, average='macro'), 4))

### Evaluasi Hasil Tuning

Kode berikut mengambil kombinasi dengan rata-rata F1 tertinggi untuk setiap penalti, melatih ulang modelnya, lalu menghitung F1 pada holdout.

In [ ]:
tuned_lr_models = {}
tuned_lr_thresholds = {}
tuned_lr_preds = {}
lr_tuning_rows = []

for penalty in lr_models:
    penalty_scores = cv_summary.loc[penalty]
    lmda, threshold = penalty_scores.idxmax()
    model = LogisticRegression(penalty=penalty, lmda=lmda, **lr_options)
    model.fit(X_train_a, y_train_a)
    pred = model.predict(X_val_a, threshold=threshold)

    tuned_lr_models[penalty] = model
    tuned_lr_thresholds[penalty] = threshold
    tuned_lr_preds[penalty] = pred
    lr_tuning_rows.append({
        'Penalti': penalty,
        'Lambda': lmda,
        'Ambang': threshold,
        'Macro F1 K-Fold': penalty_scores.max(),
        'Macro F1 holdout': f1_score(y_val, pred, zero_division=0, average='macro'),
    })

lr_tuning = pd.DataFrame(lr_tuning_rows).set_index('Penalti')
best_penalty = lr_tuning['Macro F1 K-Fold'].idxmax()
tuned_lr = tuned_lr_models[best_penalty]
best_threshold = tuned_lr_thresholds[best_penalty]
val_prob_lr = tuned_lr.probability(X_val_a)
tuned_pred_lr = tuned_lr.predict(X_val_a, threshold=best_threshold)
print('Penalti terpilih berdasarkan K-Fold:', best_penalty)
lr_tuning.round(4)

**Insights**

- *(Isi setelah dijalankan: lambda dan ambang terbaik setiap penalti, serta kenaikan Macro F1 holdout dibanding ambang 0,5.)*
- Ambang optimal Logistic Regression biasanya di bawah 0,5, karena Macro F1 memberi bobot sama pada kelas batal yang lebih sedikit.

### Pemilihan Model Terbaik

Tabel berikut merangkum konfigurasi final setiap model: Macro F1 K-Fold dari tuning (dasar pemilihan) dan Macro F1 holdout (pemeriksaan akhir). Model dengan K-Fold tertinggi dilatih ulang pada seluruh `train.csv` untuk submission.

In [ ]:
# nama -> (model, prepare, prediksi holdout, opsi predict, Macro F1 K-Fold)
final_models = {
    'KNN': (tuned_knn_kerabat, prepare_features_knn_kerabat, tuned_pred_knn_kerabat, {}, CV_KNN),
    'Decision Tree': (tuned_dtl, prepare_features_dtl_final, tuned_pred_dtl, {}, best_dtl_cv_f1),
    'Decision Tree (Alt)': (tuned_dtl_alt, prepare_features_dtl_final, tuned_pred_dtl_alt,
                            {'threshold': best_dtl_alt_threshold}, best_dtl_alt_cv_f1),
}
for penalty, name in {'l1': 'Logistic Regression (L1)', 'l2': 'Logistic Regression (L2)',
                      'elasticnet': 'Logistic Regression (Elastic Net)'}.items():
    final_models[name] = (tuned_lr_models[penalty], prepare_features_a, tuned_lr_preds[penalty],
                          {'threshold': tuned_lr_thresholds[penalty]}, float(lr_tuning.loc[penalty, 'Macro F1 K-Fold']))
final_models['SVM'] = (tuned_svm, prepare_features_pca, tuned_pred_svm, {}, float(svm_tuning['Macro F1 K-Fold'].max()))
# Naive Bayes: MAP dengan Laplace +1 (sesuai spesifikasi), fitur hasil backward elimination
final_models['Naive Bayes'] = (nb_model, prepare_features_nb, y_pred_nb, {}, nb_cv_f1)

tuned_models = {name: entry[0] for name, entry in final_models.items()}
active_models = {name: (entry[0], entry[1], entry[2]) for name, entry in final_models.items()}
model_predict_options = {name: entry[3] for name, entry in final_models.items()}
model_scores = pd.Series({name: entry[4] for name, entry in final_models.items()})

ringkasan_final = pd.DataFrame({
    'Macro F1 K-Fold': model_scores,
    'Macro F1 holdout': pd.Series({name: f1_score(y_val, entry[2], average='macro') for name, entry in final_models.items()}),
}).sort_values('Macro F1 K-Fold', ascending=False)

best_name = model_scores.idxmax()
best_model = tuned_models[best_name]
best_prepare = active_models[best_name][1]
predict_options = model_predict_options[best_name]
print('Model terpilih:', best_name)
ringkasan_final.round(4)

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

### K-Nearest Neighbors

`KNeighborsClassifier` tidak mendukung aturan beda. Jadi pembandingan dilakukan pada bagian yang setara: fitur numerik yang sama, `k`, `p`, bobot suara $1/d^q$ (lewat parameter `weights`), dan threshold yang sama, dengan semua penalti 0. Jika prediksinya identik, perhitungan jarak, voting, dan threshold kami benar, dan selisih skor terhadap model final hanya berasal dari aturan beda.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

n_fitur_knn = X_train_knn_kerabat.shape[1] - N_SIMBOL - len(TOLERANSI)   # kolom numerik (tanpa kolom aturan)
X_fit_plain, X_eval_plain = X_train_knn_kerabat[:, :n_fitur_knn], X_val_knn_kerabat[:, :n_fitur_knn]

q = Q_KNN
sk_knn_kerabat = KNeighborsClassifier(n_neighbors=K_KNN, p=P_KNN, algorithm='brute',
                                      weights=(lambda d: 1.0 / (d + 1e-9) ** q) if q > 0 else 'uniform')
sk_knn_kerabat.fit(X_fit_plain, y_train_knn_kerabat)
pred_sk_kerabat = (sk_knn_kerabat.predict_proba(X_eval_plain)[:, 1] >= THR_KNN).astype(int)

tanpa_aturan = KNNKerabat(k=K_KNN, p=P_KNN, bobot_suara=Q_KNN, threshold=THR_KNN).fit(X_fit_plain, y_train_knn_kerabat)
pred_tanpa_aturan = tanpa_aturan.predict(X_eval_plain)

pd.DataFrame([
    {'Model': 'scikit-learn (tanpa aturan beda)', 'Macro F1 holdout': f1_score(y_val, pred_sk_kerabat, average='macro')},
    {'Model': 'KNNKerabat, penalti 0', 'Macro F1 holdout': f1_score(y_val, pred_tanpa_aturan, average='macro'),
     'Prediksi sama dengan scikit-learn': (pred_sk_kerabat == pred_tanpa_aturan).mean()},
    {'Model': 'KNNKerabat final', 'Macro F1 holdout': f1_score(y_val, tuned_pred_knn_kerabat, average='macro')},
]).set_index('Model').round(4)

**Perbandingan:**

Dengan semua penalti 0, prediksi `KNNKerabat` sama 100% dengan `KNeighborsClassifier` (0,8465), sehingga perhitungan jarak, voting berbobot, dan threshold sudah benar. Kenaikan dari 0,8465 ke 0,8605 murni berasal dari aturan beda.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def to_sklearn_dt_params(params):
    allowed = ('max_depth', 'min_samples_split', 'min_samples_leaf', 'class_weight')
    return {k: v for k, v in params.items() if k in allowed}

dtl_configs = {
    f"DTL_Main tuned (depth={best_dtl_params['max_depth']}, leaf={best_dtl_params['min_samples_leaf']}, pruning={best_dtl_params['pruning']})":
        (tuned_pred_dtl, to_sklearn_dt_params(best_dtl_params), None),
    f"DTL_Alt tuned (depth={best_dtl_alt_params['max_depth']}, leaf={best_dtl_alt_params['min_samples_leaf']}, {best_dtl_alt_params['criterion']}, ambang={best_dtl_alt_threshold:g})":
        (tuned_pred_dtl_alt, to_sklearn_dt_params(best_dtl_alt_params), best_dtl_alt_threshold),
}

dtl_comparison = {}
for label, (scratch_pred, sk_params, threshold) in dtl_configs.items():
    row = {'Macro F1 kelompok': f1_score(y_val, scratch_pred, zero_division=0, average='macro')}
    for ccp_alpha in (0.0, 1e-4):
        sk_dtl = DecisionTreeClassifier(criterion='entropy', ccp_alpha=ccp_alpha, random_state=CFG.SEED, **sk_params)
        sk_dtl.fit(X_train_dtl_final, y_train_dtl_final)
        sk_pred = (sk_dtl.predict(X_val_dtl_final) if threshold is None
                   else (sk_dtl.predict_proba(X_val_dtl_final)[:, 1] >= threshold).astype(int))
        row[f'scikit-learn entropy (ccp_alpha={ccp_alpha:g})'] = f1_score(y_val, sk_pred, zero_division=0, average='macro')
    dtl_comparison[label] = row

print('Versi data:', DATA_DTL)
pd.DataFrame(dtl_comparison).T.round(4)

### Decision Tree Learning

Kode di atas membandingkan `DTL_Main` dan `DTL_Alt` hasil *tuning* dengan `DecisionTreeClassifier(criterion='entropy')` scikit-learn pada data (`prepare_features_dtl_final`), struktur pohon, dan ambang yang sama, dengan dan tanpa *cost-complexity pruning* (`ccp_alpha`).

### Logistic Regression

Kode berikut melatih logistic regression scikit-learn dengan fitur dan ambang yang sama seperti model awal kelompok. Nilai C disesuaikan dengan kekuatan regularisasi implementasi kelompok. Tabel hasil membandingkan F1 kedua implementasi untuk setiap penalti.

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression

comparison = {}
for penalty, scratch in lr_models.items():
    options = {'l1_ratio': scratch.l1_ratio} if penalty == 'elasticnet' else {}
    reference = SklearnLogisticRegression(
        penalty=penalty, C=1 / (len(y_train_a) * scratch.lmda), solver='saga',
        max_iter=3000, random_state=CFG.SEED, **options,
    ).fit(X_train_a, y_train_a)
    comparison[penalty] = {
        'Macro F1 kelompok': f1_score(y_val, lr_predictions[penalty], average='macro'),
        'Macro F1 scikit-learn': f1_score(y_val, reference.predict(X_val_a), average='macro'),
    }

pd.DataFrame(comparison).T.round(4)

**Perbandingan:**

*(Isi setelah dijalankan: selisih Macro F1 kelompok dan scikit-learn untuk setiap penalti pada ambang 0,5.)* Selisih kecil berasal dari solver yang berbeda: implementasi kelompok memakai *stochastic gradient descent* dengan jumlah iterasi tetap, sedangkan scikit-learn memakai SAGA sampai konvergen.

### Naive Bayes

Kode berikut melatih `CategoricalNB` scikit-learn dengan data, fitur, dan `alpha` yang sama seperti model kelompok, lalu membandingkan Macro F1, prediksi, dan probabilitasnya.

In [ ]:
sk_nb = CategoricalNB(alpha=1.0).fit(X_train_nb, y_train_nb)
y_pred_sk_nb = sk_nb.predict(X_val_nb)

print('Macro F1 kelompok     :', round(f1_score(y_val, y_pred_nb, average='macro'), 4))
print('Macro F1 scikit-learn :', round(f1_score(y_val, y_pred_sk_nb, average='macro'), 4))
print('Prediksi identik      :', np.array_equal(y_pred_nb, y_pred_sk_nb))
print('Selisih proba maksimum:', np.abs(nb_model.predict_proba(X_val_nb) - sk_nb.predict_proba(X_val_nb)).max())

**Perbandingan:**

Implementasi kelompok dan `CategoricalNB` menghasilkan Macro F1 yang sama dan prediksi yang identik pada seluruh data validasi. Selisih probabilitas hanya berasal dari pembulatan floating point.

### SVM

Implementasi dari awal dibandingkan dengan `SVC` scikit-learn pada konfigurasi kernel, `C`, gamma, degree, dan coef0 yang sama. Keduanya dilatih pada data training holdout yang sama, lalu prediksinya dievaluasi pada data validasi yang sama menggunakan Macro F1.

In [ ]:
from sklearn.svm import SVC

sklearn_options = {
    'C': float(best_svm_row['C']),
    'kernel': {'polynomial': 'poly'}.get(best_svm_kernel, best_svm_kernel),
    'random_state': CFG.SEED,
}
if best_svm_row['Kernel'] == 'Polynomial':
    sklearn_options.update({
        'degree': int(best_svm_row['degree']),
        'coef0': float(best_svm_row['coef0']),
        'gamma': float(best_svm_row['gamma']),
    })
elif best_svm_row['Kernel'] == 'Gaussian RBF':
    sklearn_options['gamma'] = float(best_svm_row['gamma'])

sklearn_svm = SVC(**sklearn_options).fit(X_svm_fit, y_svm_fit)
sklearn_pred_svm = sklearn_svm.predict(X_svm_val)

custom_support = len(tuned_svm.support_vectors) if tuned_svm.support_vectors is not None else 0
sklearn_support = int(np.sum(sklearn_svm.n_support_))
predictions_identical = np.mean(tuned_pred_svm == sklearn_pred_svm)

pd.DataFrame([{
    'Kernel': best_svm_row['Kernel'],
    'Macro F1 kelompok': f1_score(y_svm_val, tuned_pred_svm, average='macro'),
    'Macro F1 scikit-learn': f1_score(y_svm_val, sklearn_pred_svm, average='macro'),
    'Proporsi prediksi identik': predictions_identical,
    'Support vector kelompok': custom_support,
    'Support vector scikit-learn': sklearn_support,
    'Konvergen kelompok': tuned_svm.converged_,
}], index=['SVM']).round(4)

**Perbandingan:**

SVM implementasi kelompok dibandingkan dengan `SVC` scikit-learn menggunakan kernel dan parameter yang sama. Perbedaan Macro F1, jumlah support vector, dan proporsi prediksi yang identik menunjukkan seberapa dekat kedua solusi berada. Jika hasil prediksi identik tinggi dan Macro F1 keduanya mendekati nilai yang sama, implementasi dari awal dianggap memiliki perilaku yang konsisten meskipun solver dan toleransi optimasinya berbeda.

---

# Saving and Loading Models <a name="9"></a>

---

<!-- artifacts: <model>_v3.pkl -->

This section stores the trained models and restores them for prediction.

In [ ]:
model_files = {
    'KNN': 'knn_kerabat.npz',
    'Decision Tree': 'dtl_c45.pkl',
    'Decision Tree (Alt)': 'dtl_alt.pkl',
    'Logistic Regression (L1)': 'lr_l1.npz',
    'Logistic Regression (L2)': 'lr_l2.npz',
    'Logistic Regression (Elastic Net)': 'lr_elasticnet.npz',
    'SVM': 'svm.npz',
    'Naive Bayes': 'naive_bayes.npz',
}

loaded_models = {}
save_results = []

for name in tuned_models:
    filename = model_files.get(name)
    if filename is None:
        continue

    model = tuned_models[name]
    if not hasattr(model, 'save'):
        print(f'PERINGATAN: kelas {type(model).__name__} ({name}) belum punya method save/load.')
        continue
    path = CFG.OUTPUT_DIR / 'models' / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    model.save(path)
    loaded_model = type(model).load(path)
    loaded_models[name] = loaded_model

    prepare = active_models[name][1]
    _, _, X_check = prepare(train_raw, val_raw)
    options = model_predict_options[name]
    original_pred = model.predict(X_check, **options)
    loaded_pred = loaded_model.predict(X_check, **options)
    save_results.append({
        'Model': name,
        'Berkas': filename,
        'Prediksi sama': np.array_equal(original_pred, loaded_pred),
    })

pd.DataFrame(save_results)

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
if best_name == 'SVM':
    X_full, y_full, X_test = prepare_features_pca(train_df, test_df)
    final_model = deepcopy(best_model)
    final_model.fit(X_full, y_full)
    predictions = final_model.predict(X_test)
else:
    X_full, y_full, X_test = best_prepare(train_df, test_df)
    final_model = deepcopy(best_model)
    final_model.fit(X_full, y_full)
    predictions = final_model.predict(X_test, **predict_options)

submission = sample_sub[[ID_COL]].copy()
predictions = pd.Series(predictions, index=test_df[ID_COL])
submission[TARGET] = submission[ID_COL].map(predictions)
submission.to_csv(CFG.OUTPUT_DIR / 'submission.csv', index=False)
print('Model final:', best_name)
submission.head()

### Submission KNN dan Decision Tree

Selain `submission.csv` dari model terpilih, KNN kerabat dan `DTL_Alt` final juga disimpan sebagai `submission_knn.csv` dan `submission_dtl.csv`, supaya keduanya bisa di-submit dan dibandingkan di Kaggle. Model terpilih tidak dilatih ulang dua kali.

In [ ]:
t0 = time.time()
pred_tambahan = {}
for nama_file, nama_model in {'submission_knn.csv': 'KNN', 'submission_dtl.csv': 'Decision Tree (Alt)'}.items():
    model, prepare, _, options, _ = final_models[nama_model]
    if nama_model == best_name:
        pred = predictions.to_numpy()          # sudah dihitung oleh sel submission di atas
    else:
        X_full_k, y_full_k, X_test_k = prepare(train_df, test_df)
        pred = deepcopy(model).fit(X_full_k, y_full_k).predict(X_test_k, **options)
    pred_tambahan[nama_file] = pred
    sub = sample_sub[[ID_COL]].copy()
    sub[TARGET] = sub[ID_COL].map(pd.Series(pred, index=test_df[ID_COL].to_numpy()))
    assert sub[TARGET].notna().all(), 'Ada ID submission yang belum punya prediksi.'
    sub[TARGET] = sub[TARGET].astype(int)
    sub.to_csv(CFG.OUTPUT_DIR / nama_file, index=False)
    print(f'{nama_file}: proporsi batal {sub[TARGET].mean():.4f}')

print('Prediksi test KNN vs DTL yang berbeda:',
      int((pred_tambahan['submission_knn.csv'] != pred_tambahan['submission_dtl.csv']).sum()), 'baris')
print(f'Waktu: {(time.time() - t0) / 60:.1f} menit')

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.

### Kesalahan Model Final pada Data Validasi

Fungsi berikut menampilkan confusion matrix, contoh kesalahan dengan peluang paling meyakinkan, serta proporsi kesalahan per kanal pemesanan dan per ada/tidaknya kerabat di data latih. False positive (FP) berarti reservasi yang tetap berlangsung diprediksi batal; false negative (FN) berarti pembatalan diprediksi tetap berlangsung. Analisis dilakukan untuk KNN kerabat dan `DTL_Alt` final sebagai pembanding.

In [ ]:
punya_kerabat_val = (val_raw[KUNCI_KERABAT]
                     .merge(train_raw.groupby(KUNCI_KERABAT).size().rename('n').reset_index(), on=KUNCI_KERABAT, how='left')['n']
                     .notna().to_numpy())

def analisis_kesalahan(nama, pred, prob, warna='Blues'):
    matrix = confusion_matrix(y_val, pred, labels=[0, 1])
    sns.heatmap(matrix, annot=True, fmt='d', cmap=warna, cbar=False,
                xticklabels=['Tidak batal', 'Batal'], yticklabels=['Tidak batal', 'Batal'])
    plt.xlabel('Prediksi')
    plt.ylabel('Aktual')
    plt.title(f'{nama} pada holdout')
    plt.tight_layout()
    plt.show()
    tn, fp, fn, tp = matrix.ravel()
    print(f'{nama}: {fp + fn} dari {len(y_val)} prediksi salah ({(fp + fn) / len(y_val):.1%}). FP = {fp}, FN = {fn}.')

    hasil = val_raw[[ID_COL, 'booking_channel', 'days_before_arrival', 'nightly_rate_idr', 'special_requests', TARGET]].copy()
    hasil['Prediksi'], hasil['P(batal)'] = pred, prob
    hasil['FP'] = (hasil[TARGET] == 0) & (hasil['Prediksi'] == 1)
    hasil['FN'] = (hasil[TARGET] == 1) & (hasil['Prediksi'] == 0)
    hasil['Punya kerabat'] = punya_kerabat_val

    contoh = pd.concat([hasil[hasil['FP']].nlargest(3, 'P(batal)').assign(Kesalahan='FP'),
                        hasil[hasil['FN']].nsmallest(3, 'P(batal)').assign(Kesalahan='FN')])
    display(contoh.drop(columns=['FP', 'FN']).round(3))
    for kolom in ('booking_channel', 'Punya kerabat'):
        tabel = hasil.groupby(kolom, dropna=False).agg(Reservasi=(ID_COL, 'size'), FP=('FP', 'sum'), FN=('FN', 'sum'))
        tabel['Proporsi salah'] = (tabel['FP'] + tabel['FN']) / tabel['Reservasi']
        display(tabel.sort_values('Reservasi', ascending=False).round(3))
    return hasil

val_prob_knn_kerabat = tuned_knn_kerabat.predict_proba(X_val_knn_kerabat)[:, 1]
hasil_knn = analisis_kesalahan('KNN kerabat', tuned_pred_knn_kerabat, val_prob_knn_kerabat)

**Insights**

- *(Isi setelah dijalankan: jumlah FP dan FN KNN kerabat, kanal dengan proporsi salah tertinggi, dan perbedaan proporsi salah antara reservasi yang punya dan tidak punya kerabat.)*
- Contoh kesalahan dengan peluang paling meyakinkan biasanya adalah reservasi yang kerabatnya berlabel berbeda, sehingga tetangga terdekatnya menyesatkan.

In [ ]:
hasil_dtl_alt = analisis_kesalahan('DTL_Alt final', tuned_pred_dtl_alt, val_prob_dtl_alt, warna='Purples')

**Insights**

- *(Isi setelah dijalankan: bandingkan pola kesalahan `DTL_Alt` dengan KNN kerabat, terutama pada reservasi yang punya kerabat.)*